In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [2]:
import numpy as np
import pandas as pd
import os
import gc
import warnings

warnings.filterwarnings("ignore")

from sklearn.metrics import mean_squared_log_error
from catboost import CatBoostRegressor

SEED = 42
np.random.seed(SEED)

print("Libraries loaded")

Libraries loaded


In [3]:
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/competitions/ch-27-celebal-technologies-nit-rourkela/sample_submission.csv
/kaggle/input/competitions/ch-27-celebal-technologies-nit-rourkela/hub_metadata.csv
/kaggle/input/competitions/ch-27-celebal-technologies-nit-rourkela/orders_test.csv
/kaggle/input/competitions/ch-27-celebal-technologies-nit-rourkela/orders_train.csv


In [6]:
DATA_PATH = "/kaggle/input/competitions/ch-27-celebal-technologies-nit-rourkela"

train = pd.read_csv(
    f"{DATA_PATH}/orders_train.csv",
    parse_dates=["Date"]
)

test = pd.read_csv(
    f"{DATA_PATH}/orders_test.csv",
    parse_dates=["Date"]
)

meta = pd.read_csv(
    f"{DATA_PATH}/hub_metadata.csv"
)

sample_submission = pd.read_csv(
    f"{DATA_PATH}/sample_submission.csv"
)

print("Train shape:", train.shape)
print("Test shape:", test.shape)
print("Metadata shape:", meta.shape)
print("Submission shape:", sample_submission.shape)

print("\nTrain columns:")
print(train.columns.tolist())

print("\nTest columns:")
print(test.columns.tolist())

Train shape: (970379, 9)
Test shape: (46830, 8)
Metadata shape: (1115, 10)
Submission shape: (46830, 2)

Train columns:
['HubID', 'Weekday', 'Date', 'OrderVolume', 'AppSessions', 'IsOpen', 'PromoActive', 'RegionalHoliday', 'SchoolClosureFlag']

Test columns:
['Id', 'HubID', 'Weekday', 'Date', 'IsOpen', 'PromoActive', 'RegionalHoliday', 'SchoolClosureFlag']


In [7]:
print("TRAIN DATE RANGE:", train["Date"].min(), "→", train["Date"].max())
print("TEST DATE RANGE :", test["Date"].min(), "→", test["Date"].max())

print("\nTrain missing values:")
print(train.isnull().sum()[train.isnull().sum() > 0])

print("\nTest missing values:")
print(test.isnull().sum()[test.isnull().sum() > 0])

print("\nTarget statistics:")
print(train["OrderVolume"].describe())

TRAIN DATE RANGE: 2013-01-01 00:00:00 → 2015-06-19 00:00:00
TEST DATE RANGE : 2015-06-20 00:00:00 → 2015-07-31 00:00:00

Train missing values:
Series([], dtype: int64)

Test missing values:
Series([], dtype: int64)

Target statistics:
count    970379.000000
mean       5762.766017
std        3855.457183
min           0.000000
25%        3708.000000
50%        5736.000000
75%        7850.000000
max       38722.000000
Name: OrderVolume, dtype: float64


In [11]:
from sklearn.metrics import mean_squared_log_error

def rmsle(y_true, y_pred):
    y_pred = np.maximum(y_pred, 0)
    return np.sqrt(mean_squared_log_error(y_true, y_pred))

In [12]:
VALIDATION_DAYS = 42

validation_start = train["Date"].max() - pd.Timedelta(days=VALIDATION_DAYS - 1)

train_split = train[
    train["Date"] < validation_start
].copy()

valid_split = train[
    train["Date"] >= validation_start
].copy()

print("Training period:")
print(train_split["Date"].min(), "→", train_split["Date"].max())

print("\nValidation period:")
print(valid_split["Date"].min(), "→", valid_split["Date"].max())

print("\nTrain rows:", train_split.shape)
print("Validation rows:", valid_split.shape)

Training period:
2013-01-01 00:00:00 → 2015-05-08 00:00:00

Validation period:
2015-05-09 00:00:00 → 2015-06-19 00:00:00

Train rows: (923549, 9)
Validation rows: (46830, 9)


In [15]:
def create_statistical_predictions(train_data, predict_data):

    predict = predict_data.copy()
    global_mean = train_data["OrderVolume"].mean()

    # Most specific pattern
    cols1 = [
        "HubID",
        "Weekday",
        "PromoActive",
        "RegionalHoliday",
        "SchoolClosureFlag"
    ]

    stats1 = (
        train_data
        .groupby(cols1)["OrderVolume"]
        .mean()
        .reset_index()
        .rename(columns={"OrderVolume": "pred1"})
    )

    predict = predict.merge(
        stats1,
        on=cols1,
        how="left"
    )

    # Hub + weekday + promotion
    cols2 = [
        "HubID",
        "Weekday",
        "PromoActive"
    ]

    stats2 = (
        train_data
        .groupby(cols2)["OrderVolume"]
        .mean()
        .reset_index()
        .rename(columns={"OrderVolume": "pred2"})
    )

    predict = predict.merge(
        stats2,
        on=cols2,
        how="left"
    )

    # Hub + weekday
    cols3 = [
        "HubID",
        "Weekday"
    ]

    stats3 = (
        train_data
        .groupby(cols3)["OrderVolume"]
        .mean()
        .reset_index()
        .rename(columns={"OrderVolume": "pred3"})
    )

    predict = predict.merge(
        stats3,
        on=cols3,
        how="left"
    )

    # Hub average
    hub_avg = (
        train_data
        .groupby("HubID")["OrderVolume"]
        .mean()
        .reset_index()
        .rename(columns={"OrderVolume": "pred4"})
    )

    predict = predict.merge(
        hub_avg,
        on="HubID",
        how="left"
    )

    # Hierarchical fallback
    predict["Prediction"] = (
        predict["pred1"]
        .fillna(predict["pred2"])
        .fillna(predict["pred3"])
        .fillna(predict["pred4"])
        .fillna(global_mean)
    )

    # Closed hubs must have zero demand
    predict.loc[
        predict["IsOpen"] == 0,
        "Prediction"
    ] = 0

    return predict["Prediction"].values

In [14]:
baseline_predictions = create_statistical_predictions(
    train_split,
    valid_split
)

baseline_score = rmsle(
    valid_split["OrderVolume"],
    baseline_predictions
)

print("=" * 50)
print("BASELINE RMSLE:", baseline_score)
print("=" * 50)

BASELINE RMSLE: 0.15433192663869866


In [16]:
def create_lag_features(df):

    df = df.copy()
    df = df.sort_values(["HubID", "Date"])

    # Lags
    lags = [1, 7, 14, 21, 28, 35, 42]

    for lag in lags:
        df[f"lag_{lag}"] = (
            df.groupby("HubID")["OrderVolume"]
            .shift(lag)
        )

    # Rolling averages (shifted to avoid leakage)
    windows = [7, 14, 28, 56]

    for window in windows:
        df[f"rolling_mean_{window}"] = (
            df.groupby("HubID")["OrderVolume"]
            .transform(
                lambda x: x.shift(1).rolling(window, min_periods=3).mean()
            )
        )

        df[f"rolling_std_{window}"] = (
            df.groupby("HubID")["OrderVolume"]
            .transform(
                lambda x: x.shift(1).rolling(window, min_periods=3).std()
            )
        )

    return df

In [18]:
def create_base_features(df, meta):

    df = df.copy()

    # Merge metadata
    df = df.merge(meta, on="HubID", how="left")

    # Date features
    df["Year"] = df["Date"].dt.year
    df["Month"] = df["Date"].dt.month
    df["Day"] = df["Date"].dt.day
    df["DayOfYear"] = df["Date"].dt.dayofyear
    df["WeekOfYear"] = df["Date"].dt.isocalendar().week.astype(int)

    df["IsWeekend"] = (df["Date"].dt.dayofweek >= 5).astype(int)
    df["IsMonthStart"] = df["Date"].dt.is_month_start.astype(int)
    df["IsMonthEnd"] = df["Date"].dt.is_month_end.astype(int)

    # Cyclical features
    df["Month_sin"] = np.sin(2 * np.pi * df["Month"] / 12)
    df["Month_cos"] = np.cos(2 * np.pi * df["Month"] / 12)

    df["Weekday_sin"] = np.sin(2 * np.pi * df["Weekday"] / 7)
    df["Weekday_cos"] = np.cos(2 * np.pi * df["Weekday"] / 7)

    # Competitor age
    df["CompetitorAge"] = (
        df["Year"] - df["CompetitorOpenSinceYear"]
    )

    # Loyalty age
    df["LoyaltyAge"] = (
        df["Year"] - df["LoyaltyProgramSinceYear"]
    )

    return df

In [19]:
lag_train = create_lag_features(train_split)

lag_train = create_base_features(
    lag_train,
    meta
)

print(lag_train.shape)

display(lag_train.head())

(923549, 47)


,HubID,Weekday,Date,OrderVolume,AppSessions,IsOpen,PromoActive,RegionalHoliday,SchoolClosureFlag,lag_1,...,WeekOfYear,IsWeekend,IsMonthStart,IsMonthEnd,Month_sin,Month_cos,Weekday_sin,Weekday_cos,CompetitorAge,LoyaltyAge
0,1,2,2013-01-01,0,0,0,0,1,1,NaN,...,1,0,1,0,0.5,0.866025,0.974928,-0.222521,5.0,NaN
1,1,3,2013-01-02,5530,668,1,0,0,1,0.0,...,1,0,0,0,0.5,0.866025,0.433884,-0.900969,5.0,NaN
2,1,4,2013-01-03,4327,578,1,0,0,1,5530.0,...,1,0,0,0,0.5,0.866025,-0.433884,-0.900969,5.0,NaN
3,1,5,2013-01-04,4486,619,1,0,0,1,4327.0,...,1,0,0,0,0.5,0.866025,-0.974928,-0.222521,5.0,NaN
4,1,6,2013-01-05,4997,635,1,0,0,1,4486.0,...,1,1,0,0,0.5,0.866025,-0.781831,0.623490,5.0,NaN


In [20]:
combined = pd.concat(
    [train_split, valid_split],
    axis=0,
    ignore_index=True
)

combined = combined.sort_values(
    ["HubID", "Date"]
)

combined_lag = create_lag_features(combined)

combined_lag = create_base_features(
    combined_lag,
    meta
)

lag_train = combined_lag[
    combined_lag["Date"] < validation_start
].copy()

lag_valid = combined_lag[
    combined_lag["Date"] >= validation_start
].copy()

print("Lag train:", lag_train.shape)
print("Lag valid:", lag_valid.shape)

Lag train: (923549, 47)
Lag valid: (46830, 47)


In [21]:
DROP_COLUMNS = [
    "Date",
    "OrderVolume",
    "AppSessions"
]

FEATURES = [
    col for col in lag_train.columns
    if col not in DROP_COLUMNS
]

CAT_FEATURES = [
    "HubID",
    "HubFormat",
    "AssortmentTier",
    "LoyaltyProgram",
    "LoyaltyProgramInterval"
]

CAT_FEATURES = [
    col for col in CAT_FEATURES
    if col in FEATURES
]

print("Number of features:", len(FEATURES))
print("\nCategorical features:")
print(CAT_FEATURES)

Number of features: 44

Categorical features:
['HubID', 'HubFormat', 'AssortmentTier', 'LoyaltyProgram', 'LoyaltyProgramInterval']


In [22]:
for col in CAT_FEATURES:

    lag_train[col] = (
        lag_train[col]
        .fillna("Missing")
        .astype(str)
    )

    lag_valid[col] = (
        lag_valid[col]
        .fillna("Missing")
        .astype(str)
    )


for col in FEATURES:

    if col not in CAT_FEATURES:

        lag_train[col] = (
            lag_train[col]
            .replace([np.inf, -np.inf], np.nan)
            .fillna(-999)
        )

        lag_valid[col] = (
            lag_valid[col]
            .replace([np.inf, -np.inf], np.nan)
            .fillna(-999)
        )

In [23]:
from catboost import CatBoostRegressor

model_lag = CatBoostRegressor(

    loss_function="RMSE",

    iterations=1200,

    learning_rate=0.05,

    depth=9,

    l2_leaf_reg=5,

    random_seed=42,

    verbose=100,

    thread_count=-1,

    od_type="Iter",

    od_wait=100
)


model_lag.fit(

    lag_train[FEATURES],

    np.log1p(lag_train["OrderVolume"]),

    cat_features=CAT_FEATURES,

    eval_set=(

        lag_valid[FEATURES],

        np.log1p(lag_valid["OrderVolume"])

    ),

    early_stopping_rounds=100,

    verbose=100
)

0:	learn: 3.1470804	test: 3.3732082	best: 3.3732082 (0)	total: 727ms	remaining: 14m 32s
100:	learn: 0.1367391	test: 0.1316950	best: 0.1316950 (100)	total: 1m 19s	remaining: 14m 20s
200:	learn: 0.1171254	test: 0.1182450	best: 0.1182307 (199)	total: 2m 40s	remaining: 13m 19s
300:	learn: 0.1098430	test: 0.1158696	best: 0.1158498 (295)	total: 4m 3s	remaining: 12m 7s
400:	learn: 0.1059922	test: 0.1142622	best: 0.1142433 (397)	total: 5m 25s	remaining: 10m 48s
500:	learn: 0.1030992	test: 0.1137540	best: 0.1136748 (490)	total: 6m 51s	remaining: 9m 34s
600:	learn: 0.1008456	test: 0.1132320	best: 0.1132320 (600)	total: 8m 16s	remaining: 8m 14s
700:	learn: 0.0990531	test: 0.1129529	best: 0.1129314 (669)	total: 9m 42s	remaining: 6m 54s
800:	learn: 0.0974944	test: 0.1129409	best: 0.1128387 (731)	total: 11m 8s	remaining: 5m 32s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 0.1128386664
bestIteration = 731

Shrink model to first 732 iterations.


CatBoostRegressor(depth=9, iterations=1200, l2_leaf_reg=5, learning_rate=0.05, loss_function='RMSE', od_type='Iter', od_wait=100, random_seed=42, verbose=100)

In [24]:
lag_pred_log = model_lag.predict(
    lag_valid[FEATURES]
)

lag_predictions = np.expm1(
    lag_pred_log
)

lag_predictions = np.maximum(
    lag_predictions,
    0
)

# Closed hubs = zero
lag_predictions[
    lag_valid["IsOpen"].values == 0
] = 0


lag_score = rmsle(
    lag_valid["OrderVolume"],
    lag_predictions
)


print("=" * 50)
print("BASELINE RMSLE :", baseline_score)
print("LAG MODEL RMSLE:", lag_score)
print("=" * 50)

BASELINE RMSLE : 0.15433192663869866
LAG MODEL RMSLE: 0.10842495839322677


In [25]:
DROP_COLUMNS_V2 = [
    "Date",
    "OrderVolume"
]

FEATURES_V2 = [
    col for col in lag_train.columns
    if col not in DROP_COLUMNS_V2
]

print("Number of features:", len(FEATURES_V2))
print("AppSessions included:", "AppSessions" in FEATURES_V2)

print("\nCategorical features:")
print(CAT_FEATURES)

Number of features: 45
AppSessions included: True

Categorical features:
['HubID', 'HubFormat', 'AssortmentTier', 'LoyaltyProgram', 'LoyaltyProgramInterval']


In [26]:
model_v2 = CatBoostRegressor(
    loss_function="RMSE",
    iterations=1000,
    learning_rate=0.06,
    depth=9,
    l2_leaf_reg=5,
    random_seed=42,
    verbose=100,
    thread_count=-1,
    od_type="Iter",
    od_wait=75
)

model_v2.fit(
    lag_train[FEATURES_V2],
    np.log1p(lag_train["OrderVolume"]),
    
    cat_features=CAT_FEATURES,
    
    eval_set=(
        lag_valid[FEATURES_V2],
        np.log1p(lag_valid["OrderVolume"])
    ),
    
    early_stopping_rounds=75,
    verbose=100
)

0:	learn: 3.1139842	test: 3.3380311	best: 3.3380311 (0)	total: 759ms	remaining: 12m 38s
100:	learn: 0.0950801	test: 0.1413772	best: 0.1413772 (100)	total: 1m 18s	remaining: 11m 36s
200:	learn: 0.0774189	test: 0.1274551	best: 0.1274551 (200)	total: 2m 35s	remaining: 10m 19s
300:	learn: 0.0686051	test: 0.1224499	best: 0.1224499 (300)	total: 4m	remaining: 9m 18s
400:	learn: 0.0636639	test: 0.1200091	best: 0.1200091 (400)	total: 5m 25s	remaining: 8m 6s
500:	learn: 0.0610097	test: 0.1185678	best: 0.1185641 (497)	total: 6m 48s	remaining: 6m 46s
600:	learn: 0.0587790	test: 0.1173005	best: 0.1173005 (598)	total: 8m 8s	remaining: 5m 24s
700:	learn: 0.0571534	test: 0.1162071	best: 0.1161552 (697)	total: 9m 29s	remaining: 4m 2s
800:	learn: 0.0558413	test: 0.1152809	best: 0.1152503 (796)	total: 10m 50s	remaining: 2m 41s
900:	learn: 0.0546683	test: 0.1145887	best: 0.1145887 (900)	total: 12m 10s	remaining: 1m 20s
999:	learn: 0.0538351	test: 0.1139405	best: 0.1139319 (987)	total: 13m 27s	remaining: 0

CatBoostRegressor(depth=9, iterations=1000, l2_leaf_reg=5, learning_rate=0.06, loss_function='RMSE', od_type='Iter', od_wait=75, random_seed=42, verbose=100)

In [27]:
v2_pred_log = model_v2.predict(
    lag_valid[FEATURES_V2]
)

v2_predictions = np.maximum(
    np.expm1(v2_pred_log),
    0
)

# Force closed hubs to zero
v2_predictions[
    lag_valid["IsOpen"].values == 0
] = 0

v2_score = rmsle(
    lag_valid["OrderVolume"],
    v2_predictions
)

print("=" * 55)
print("BASELINE RMSLE :", baseline_score)
print("MODEL V1 RMSLE :", lag_score)
print("MODEL V2 RMSLE :", v2_score)
print("=" * 55)

BASELINE RMSLE : 0.15433192663869866
MODEL V1 RMSLE : 0.10842495839322677
MODEL V2 RMSLE : 0.06234847454564093


In [32]:
LAGS = [1, 7, 14, 21, 28, 35, 42]
WINDOWS = [7, 14, 28, 56]

def create_features_for_forecast_day(day_data, history):

    day_data = day_data.copy()

    # =====================================================
    # LAG FEATURES
    # =====================================================

    history_sorted = history.sort_values(
        ["HubID", "Date"]
    )

    for lag in LAGS:

        lag_values = (
            history_sorted
            .groupby("HubID")["OrderVolume"]
            .nth(-lag)
        )

        day_data[f"lag_{lag}"] = (
            day_data["HubID"].map(lag_values)
        )

    # =====================================================
    # ROLLING FEATURES
    # =====================================================

    for window in WINDOWS:

        rolling_mean = (
            history_sorted
            .groupby("HubID")["OrderVolume"]
            .apply(lambda x: x.tail(window).mean())
        )

        rolling_std = (
            history_sorted
            .groupby("HubID")["OrderVolume"]
            .apply(lambda x: x.tail(window).std())
        )

        day_data[f"rolling_mean_{window}"] = (
            day_data["HubID"].map(rolling_mean)
        )

        day_data[f"rolling_std_{window}"] = (
            day_data["HubID"].map(rolling_std)
        )

    # =====================================================
    # METADATA
    # =====================================================

    day_data = day_data.merge(
        meta,
        on="HubID",
        how="left"
    )

    # =====================================================
    # DATE FEATURES
    # =====================================================

    day_data["Year"] = day_data["Date"].dt.year
    day_data["Month"] = day_data["Date"].dt.month
    day_data["Day"] = day_data["Date"].dt.day
    day_data["DayOfYear"] = day_data["Date"].dt.dayofyear

    day_data["WeekOfYear"] = (
        day_data["Date"]
        .dt
        .isocalendar()
        .week
        .astype(int)
    )

    day_data["IsWeekend"] = (
        day_data["Date"].dt.dayofweek >= 5
    ).astype(int)

    day_data["IsMonthStart"] = (
        day_data["Date"].dt.is_month_start
    ).astype(int)

    day_data["IsMonthEnd"] = (
        day_data["Date"].dt.is_month_end
    ).astype(int)

    # Cyclical features

    day_data["Month_sin"] = np.sin(
        2 * np.pi * day_data["Month"] / 12
    )

    day_data["Month_cos"] = np.cos(
        2 * np.pi * day_data["Month"] / 12
    )

    day_data["Weekday_sin"] = np.sin(
        2 * np.pi * day_data["Weekday"] / 7
    )

    day_data["Weekday_cos"] = np.cos(
        2 * np.pi * day_data["Weekday"] / 7
    )

    # =====================================================
    # AGE FEATURES
    # =====================================================

    day_data["CompetitorAge"] = (
        day_data["Year"]
        - day_data["CompetitorOpenSinceYear"]
    )

    day_data["LoyaltyAge"] = (
        day_data["Year"]
        - day_data["LoyaltyProgramSinceYear"]
    )

    return day_data

In [34]:
print("V1 features:", len(FEATURES))
print("AppSessions in V1:", "AppSessions" in FEATURES)

test_sorted = test.sort_values("Date").copy()

forecast_history = train.copy()

all_predictions = []

test_dates = sorted(test_sorted["Date"].unique())

print("\nForecast days:", len(test_dates))
print("Start:", test_dates[0])
print("End:", test_dates[-1])

V1 features: 44
AppSessions in V1: False

Forecast days: 42
Start: 2015-06-20 00:00:00
End: 2015-07-31 00:00:00


In [36]:
all_predictions = []

# Reset history
forecast_history = train.copy()

for i, current_date in enumerate(test_dates):

    current_day = test_sorted[
        test_sorted["Date"] == current_date
    ].copy()

    # Create features
    current_features = create_features_for_forecast_day(
        current_day,
        forecast_history
    )

    # Clean categorical features
    for col in CAT_FEATURES:
        current_features[col] = (
            current_features[col]
            .fillna("Missing")
            .astype(str)
        )

    # Clean numerical features
    for col in FEATURES:
        if col not in CAT_FEATURES:
            current_features[col] = (
                current_features[col]
                .replace([np.inf, -np.inf], np.nan)
                .fillna(-999)
            )

    # Check features
    missing_features = set(FEATURES) - set(current_features.columns)

    if missing_features:
        raise ValueError(
            f"Missing features: {missing_features}"
        )

    # Predict
    pred_log = model_lag.predict(
        current_features[FEATURES]
    )

    predictions = np.maximum(
        np.expm1(pred_log),
        0
    )

    # Closed hubs = zero
    predictions[
        current_features["IsOpen"].values == 0
    ] = 0

    # Save predictions
    current_day["Prediction"] = predictions

    all_predictions.append(
        current_day[["Id", "Prediction"]].copy()
    )

    # -------------------------------------------------
    # Add predicted day into history
    # -------------------------------------------------

    history_add = current_day.copy()

    history_add["OrderVolume"] = predictions

    # AppSessions doesn't exist in test, but train history
    # has this column. V1 does NOT use it, so fill placeholder.
    history_add["AppSessions"] = 0

    # Make columns exactly match training history
    history_add = history_add.reindex(
        columns=forecast_history.columns,
        fill_value=0
    )

    forecast_history = pd.concat(
        [
            forecast_history,
            history_add
        ],
        ignore_index=True
    )

    if (i + 1) % 5 == 0 or i == 0:
        print(
            f"Completed {i+1}/{len(test_dates)} days"
        )

Completed 1/42 days
Completed 5/42 days
Completed 10/42 days
Completed 15/42 days
Completed 20/42 days
Completed 25/42 days
Completed 30/42 days
Completed 35/42 days
Completed 40/42 days


In [37]:
submission = pd.concat(
    all_predictions,
    ignore_index=True
)

submission.columns = ["Id", "OrderVolume"]

# Ensure correct row order
submission = sample_submission[["Id"]].merge(
    submission,
    on="Id",
    how="left"
)

# Safety checks
submission["OrderVolume"] = (
    submission["OrderVolume"]
    .clip(lower=0)
)

print("Submission shape:", submission.shape)

print("\nMissing predictions:")
print(submission["OrderVolume"].isnull().sum())

display(submission.head())

submission.to_csv(
    "/kaggle/working/submission_v2.csv",
    index=False
)

print("\nFile saved!")

Submission shape: (46830, 2)

Missing predictions:
0


,Id,OrderVolume
0,1,3340.773377
1,2,3251.972618
2,3,4485.368869
3,4,5965.316136
4,5,3544.069295



File saved!


In [38]:
print(submission.describe())

print("\nSample submission:")
display(submission.head(10))

print("\nFile location:")
print("/kaggle/working/submission_v2.csv")

                Id   OrderVolume
count  46830.00000  46830.000000
mean   23415.50000   2704.112247
std    13518.80089   1894.448852
min        1.00000      0.000000
25%    11708.25000   1317.689301
50%    23415.50000   2641.672711
75%    35122.75000   3869.214943
max    46830.00000  12586.168323

Sample submission:


,Id,OrderVolume
0,1,3340.773377
1,2,3251.972618
2,3,4485.368869
3,4,5965.316136
4,5,3544.069295
5,6,3395.218656
6,7,5652.681286
7,8,3973.186362
8,9,4449.626146
9,10,3554.180980



File location:
/kaggle/working/submission_v2.csv


In [40]:
submission.to_csv(
    "/kaggle/working/submission_v1_recursive.csv",
    index=False
)

In [41]:
import numpy as np
import pandas as pd

# Work only with actual training data
hist = train.copy()

# Add recency weight
max_date = hist["Date"].max()

hist["days_ago"] = (
    max_date - hist["Date"]
).dt.days

# Recent observations get higher weight
hist["weight"] = np.exp(
    -hist["days_ago"] / 180
)

print(hist[["days_ago", "weight"]].describe())

            days_ago         weight
count  970379.000000  970379.000000
mean      455.916164       0.197668
std       261.648433       0.250402
min         0.000000       0.006775
25%       226.000000       0.022621
50%       464.000000       0.075943
75%       682.000000       0.284918
max       899.000000       1.000000


In [42]:
def weighted_average(group):
    return np.average(
        group["OrderVolume"],
        weights=group["weight"]
    )

hub_weekday = (
    hist
    .groupby(["HubID", "Weekday"])
    .apply(weighted_average)
    .reset_index(name="pred_hub_weekday")
)

hub_weekday_promo = (
    hist
    .groupby(["HubID", "Weekday", "PromoActive"])
    .apply(weighted_average)
    .reset_index(name="pred_hub_weekday_promo")
)

hub_weekday_context = (
    hist
    .groupby([
        "HubID",
        "Weekday",
        "PromoActive",
        "RegionalHoliday",
        "SchoolClosureFlag"
    ])
    .apply(weighted_average)
    .reset_index(name="pred_context")
)

hub_avg = (
    hist
    .groupby("HubID")
    .apply(weighted_average)
    .reset_index(name="pred_hub_avg")
)

print("Done building statistics")

Done building statistics


In [43]:
recent_start = train["Date"].max() - pd.Timedelta(days=56)

recent_hist = train[
    train["Date"] >= recent_start
].copy()

recent_hub_weekday = (
    recent_hist
    .groupby(["HubID", "Weekday"])["OrderVolume"]
    .mean()
    .reset_index()
    .rename(columns={
        "OrderVolume": "pred_recent"
    })
)

print(
    "Recent history:",
    recent_hist["Date"].min(),
    "to",
    recent_hist["Date"].max()
)

Recent history: 2015-04-24 00:00:00 to 2015-06-19 00:00:00


In [44]:
test_v2 = test.copy()

# Most specific context
test_v2 = test_v2.merge(
    hub_weekday_context,
    on=[
        "HubID",
        "Weekday",
        "PromoActive",
        "RegionalHoliday",
        "SchoolClosureFlag"
    ],
    how="left"
)

# Hub + weekday + promo
test_v2 = test_v2.merge(
    hub_weekday_promo,
    on=[
        "HubID",
        "Weekday",
        "PromoActive"
    ],
    how="left"
)

# Hub + weekday
test_v2 = test_v2.merge(
    hub_weekday,
    on=[
        "HubID",
        "Weekday"
    ],
    how="left"
)

# Recent hub + weekday
test_v2 = test_v2.merge(
    recent_hub_weekday,
    on=[
        "HubID",
        "Weekday"
    ],
    how="left"
)

# Hub average
test_v2 = test_v2.merge(
    hub_avg,
    on="HubID",
    how="left"
)

# Hierarchical prediction
test_v2["OrderVolume"] = (
    test_v2["pred_context"]
    .fillna(test_v2["pred_hub_weekday_promo"])
    .fillna(test_v2["pred_recent"])
    .fillna(test_v2["pred_hub_weekday"])
    .fillna(test_v2["pred_hub_avg"])
    .fillna(train["OrderVolume"].mean())
)

# Blend recent and long-term where both available
mask = (
    test_v2["pred_recent"].notna() &
    test_v2["pred_hub_weekday"].notna()
)

test_v2.loc[mask, "OrderVolume"] = (
    0.7 * test_v2.loc[mask, "pred_recent"] +
    0.3 * test_v2.loc[mask, "pred_hub_weekday"]
)

# Closed hubs
test_v2.loc[
    test_v2["IsOpen"] == 0,
    "OrderVolume"
] = 0

submission_v2 = test_v2[
    ["Id", "OrderVolume"]
].copy()

submission_v2 = sample_submission[
    ["Id"]
].merge(
    submission_v2,
    on="Id",
    how="left"
)

submission_v2.to_csv(
    "/kaggle/working/submission_statistical_v2.csv",
    index=False
)

print(submission_v2.shape)
print(submission_v2["OrderVolume"].describe())

(46830, 2)
count    46830.000000
mean      5951.423913
std       3410.243114
min          0.000000
25%       4442.097406
50%       6089.052439
75%       7798.995325
max      29891.003836
Name: OrderVolume, dtype: float64


In [46]:
submission_v2.to_csv(
    "/kaggle/working/submission_statistical_v2.csv",
    index=False
)

print("File saved successfully!")

import os
print("\nFiles in working directory:")
for file in os.listdir("/kaggle/working"):
    print(file)

File saved successfully!

Files in working directory:
submission_v2.csv
catboost_info
submission_v1_recursive.csv
.virtual_documents
submission_statistical_v2.csv


In [47]:
print("Train date range:", train.Date.min(), "to", train.Date.max())

for year in [2013, 2014, 2015]:
    temp = train[train.Date.dt.year == year]
    
    print(
        year,
        temp.Date.min(),
        "to",
        temp.Date.max(),
        "rows:",
        len(temp)
    )

Train date range: 2013-01-01 00:00:00 to 2015-06-19 00:00:00
2013 2013-01-01 00:00:00 to 2013-12-31 00:00:00 rows: 406974
2014 2014-01-01 00:00:00 to 2014-12-31 00:00:00 rows: 373855
2015 2015-01-01 00:00:00 to 2015-06-19 00:00:00 rows: 189550


In [48]:
# Check if all hubs exist throughout history

hub_counts = train.groupby("HubID")["Date"].agg(
    ["min", "max", "count"]
)

print(hub_counts.describe())

print("\nHubs with data in 2014:")
print(
    train[train.Date.dt.year == 2014]["HubID"].nunique()
)

print("\nTotal hubs:")
print(train["HubID"].nunique())

                                 min                  max        count
count                           1115                 1115  1115.000000
mean   2013-01-01 00:01:17.488789248  2015-06-19 00:00:00   870.295067
min              2013-01-01 00:00:00  2015-06-19 00:00:00   716.000000
25%              2013-01-01 00:00:00  2015-06-19 00:00:00   900.000000
50%              2013-01-01 00:00:00  2015-06-19 00:00:00   900.000000
75%              2013-01-01 00:00:00  2015-06-19 00:00:00   900.000000
max              2013-01-02 00:00:00  2015-06-19 00:00:00   900.000000
std                              NaN                  NaN    67.729422

Hubs with data in 2014:
1115

Total hubs:
1115


In [49]:
from sklearn.metrics import mean_squared_log_error
import numpy as np

# Simulated competition setup
VAL_START = pd.Timestamp("2014-06-20")
VAL_END = pd.Timestamp("2014-07-31")

backtrain = train[
    train["Date"] < VAL_START
].copy()

backval = train[
    (train["Date"] >= VAL_START) &
    (train["Date"] <= VAL_END)
].copy()

print("Backtrain:", backtrain.shape)
print("Backval:", backval.shape)

print("\nBacktrain end:", backtrain["Date"].max())
print("Backval range:",
      backval["Date"].min(),
      "to",
      backval["Date"].max())

Backtrain: (596524, 9)
Backval: (41250, 9)

Backtrain end: 2014-06-19 00:00:00
Backval range: 2014-06-20 00:00:00 to 2014-07-31 00:00:00


In [52]:
def add_prediction_signals(history, future):

    future = future.copy()

    max_hist_date = history["Date"].max()

    # ------------------------------------------------
    # 1. Long-term Hub + Weekday
    # ------------------------------------------------

    hub_weekday = (
        history
        .groupby(["HubID", "Weekday"])["OrderVolume"]
        .mean()
        .reset_index(name="long_hub_weekday")
    )

    future = future.merge(
        hub_weekday,
        on=["HubID", "Weekday"],
        how="left"
    )

    # ------------------------------------------------
    # 2. Recent 56 days
    # ------------------------------------------------

    recent56 = history[
        history["Date"] >= max_hist_date - pd.Timedelta(days=56)
    ]

    recent56_stats = (
        recent56
        .groupby(["HubID", "Weekday"])["OrderVolume"]
        .mean()
        .reset_index(name="recent56")
    )

    future = future.merge(
        recent56_stats,
        on=["HubID", "Weekday"],
        how="left"
    )

    # ------------------------------------------------
    # 3. Recent 112 days
    # ------------------------------------------------

    recent112 = history[
        history["Date"] >= max_hist_date - pd.Timedelta(days=112)
    ]

    recent112_stats = (
        recent112
        .groupby(["HubID", "Weekday"])["OrderVolume"]
        .mean()
        .reset_index(name="recent112")
    )

    future = future.merge(
        recent112_stats,
        on=["HubID", "Weekday"],
        how="left"
    )

    # ------------------------------------------------
    # 4. Hub overall average
    # ------------------------------------------------

    hub_avg = (
        history
        .groupby("HubID")["OrderVolume"]
        .mean()
        .reset_index(name="hub_avg")
    )

    future = future.merge(
        hub_avg,
        on="HubID",
        how="left"
    )

    # ------------------------------------------------
    # 5. Previous year same date
    # ------------------------------------------------

    prev_year = history[
        (history["Date"].dt.year == future["Date"].dt.year.min() - 1)
    ][["HubID", "Date", "OrderVolume"]].copy()

    prev_year["Date"] = (
        prev_year["Date"] + pd.DateOffset(years=1)
    )

    prev_year = prev_year.rename(
        columns={"OrderVolume": "prev_year_same_date"}
    )

    future = future.merge(
        prev_year,
        on=["HubID", "Date"],
        how="left"
    )

    return future

In [53]:
backtest = add_prediction_signals(
    backtrain,
    backval.drop(columns=["OrderVolume"])
)

print(backtest[
    [
        "long_hub_weekday",
        "recent56",
        "recent112",
        "hub_avg",
        "prev_year_same_date"
    ]
].isnull().sum())

long_hub_weekday       0
recent56               0
recent112              0
hub_avg                0
prev_year_same_date    0
dtype: int64


In [54]:
signals = [
    "long_hub_weekday",
    "recent56",
    "recent112",
    "hub_avg",
    "prev_year_same_date"
]

for col in signals:

    pred = backtest[col].fillna(
        backtest["hub_avg"]
    )

    pred = pred.clip(lower=0)

    score = np.sqrt(
        mean_squared_log_error(
            backval["OrderVolume"],
            pred
        )
    )

    print(f"{col}: {score:.6f}")

long_hub_weekday: 0.603052
recent56: 0.604517
recent112: 0.602043
hub_avg: 3.274947
prev_year_same_date: 4.648819


In [55]:
print("Backval shape:", backval.shape)
print("Backtest shape:", backtest.shape)

print("\nFirst 10 IDs/Hubs/Dates:")

check = pd.DataFrame({
    "actual_hub": backval["HubID"].values[:10],
    "actual_date": backval["Date"].values[:10],
    "actual_order": backval["OrderVolume"].values[:10],
    
    "pred_hub": backtest["HubID"].values[:10],
    "pred_date": backtest["Date"].values[:10],
    "recent56": backtest["recent56"].values[:10]
})

display(check)

print("\nHub alignment:")
print(
    (backval["HubID"].values == backtest["HubID"].values).mean()
)

print("\nDate alignment:")
print(
    (backval["Date"].values == backtest["Date"].values).mean()
)

Backval shape: (41250, 9)
Backtest shape: (41250, 13)

First 10 IDs/Hubs/Dates:


,actual_hub,actual_date,actual_order,pred_hub,pred_date,recent56
0,1,2014-06-20,6362,1,2014-06-20,4854.125
1,2,2014-06-20,4572,2,2014-06-20,5107.125
2,3,2014-06-20,9752,3,2014-06-20,7370.625
3,4,2014-06-20,9503,4,2014-06-20,10210.625
4,5,2014-06-20,4741,5,2014-06-20,5183.000
5,6,2014-06-20,5661,6,2014-06-20,5660.250
6,7,2014-06-20,9885,7,2014-06-20,10205.250
7,8,2014-06-20,5236,8,2014-06-20,5101.875
8,9,2014-06-20,9912,9,2014-06-20,6831.875
9,10,2014-06-20,5656,10,2014-06-20,6139.875



Hub alignment:
1.0

Date alignment:
1.0


In [56]:
from sklearn.metrics import mean_squared_log_error
import numpy as np
import pandas as pd

def weighted_avg(group):
    return np.average(
        group["OrderVolume"],
        weights=group["weight"]
    )


def statistical_forecast(history, future):

    hist = history.copy()
    future = future.copy()

    max_date = hist["Date"].max()

    hist["days_ago"] = (
        max_date - hist["Date"]
    ).dt.days

    hist["weight"] = np.exp(
        -hist["days_ago"] / 180
    )

    # Most detailed context
    context = (
        hist.groupby([
            "HubID",
            "Weekday",
            "PromoActive",
            "RegionalHoliday",
            "SchoolClosureFlag"
        ])
        .apply(weighted_avg)
        .reset_index(name="pred_context")
    )

    # Hub + weekday + promo
    hub_weekday_promo = (
        hist.groupby([
            "HubID",
            "Weekday",
            "PromoActive"
        ])
        .apply(weighted_avg)
        .reset_index(name="pred_hub_weekday_promo")
    )

    # Hub + weekday
    hub_weekday = (
        hist.groupby([
            "HubID",
            "Weekday"
        ])
        .apply(weighted_avg)
        .reset_index(name="pred_hub_weekday")
    )

    # Recent 56 days
    recent_start = (
        max_date - pd.Timedelta(days=56)
    )

    recent_hist = hist[
        hist["Date"] >= recent_start
    ]

    recent = (
        recent_hist
        .groupby([
            "HubID",
            "Weekday"
        ])["OrderVolume"]
        .mean()
        .reset_index(name="pred_recent")
    )

    # Hub average
    hub_avg = (
        hist.groupby("HubID")
        .apply(weighted_avg)
        .reset_index(name="pred_hub_avg")
    )

    # Merge signals
    future = future.merge(
        context,
        on=[
            "HubID",
            "Weekday",
            "PromoActive",
            "RegionalHoliday",
            "SchoolClosureFlag"
        ],
        how="left"
    )

    future = future.merge(
        hub_weekday_promo,
        on=[
            "HubID",
            "Weekday",
            "PromoActive"
        ],
        how="left"
    )

    future = future.merge(
        hub_weekday,
        on=[
            "HubID",
            "Weekday"
        ],
        how="left"
    )

    future = future.merge(
        recent,
        on=[
            "HubID",
            "Weekday"
        ],
        how="left"
    )

    future = future.merge(
        hub_avg,
        on="HubID",
        how="left"
    )

    # Base hierarchical forecast
    pred = (
        future["pred_context"]
        .fillna(future["pred_hub_weekday_promo"])
        .fillna(future["pred_recent"])
        .fillna(future["pred_hub_weekday"])
        .fillna(future["pred_hub_avg"])
        .fillna(hist["OrderVolume"].mean())
    )

    # Same blend used in submitted V2
    mask = (
        future["pred_recent"].notna()
        & future["pred_hub_weekday"].notna()
    )

    pred.loc[mask] = (
        0.7 * future.loc[mask, "pred_recent"]
        + 0.3 * future.loc[mask, "pred_hub_weekday"]
    )

    pred = pred.clip(lower=0)

    pred.loc[
        future["IsOpen"].values == 0
    ] = 0

    return pred, future


# Exact simulated competition split
VAL_START = pd.Timestamp("2014-06-20")
VAL_END = pd.Timestamp("2014-07-31")

hist_val = train[
    train["Date"] < VAL_START
].copy()

future_val = train[
    (train["Date"] >= VAL_START)
    & (train["Date"] <= VAL_END)
].copy()

actual = future_val["OrderVolume"].values

pred, debug_val = statistical_forecast(
    hist_val,
    future_val.drop(columns=["OrderVolume"])
)

score = np.sqrt(
    mean_squared_log_error(
        actual,
        pred
    )
)

print("=" * 50)
print("REALISTIC V2 BACKTEST RMSLE:", score)
print("=" * 50)

print("\nPrediction mean:", pred.mean())
print("Actual mean:", actual.mean())

REALISTIC V2 BACKTEST RMSLE: 0.2686154240232423

Prediction mean: 5774.604993245677
Actual mean: 5851.209503030303


In [57]:
# Compare recent demand with historical demand hub-by-hub

recent_56 = train[
    train["Date"] >= train["Date"].max() - pd.Timedelta(days=56)
]

recent_avg = (
    recent_56.groupby("HubID")["OrderVolume"]
    .mean()
)

long_avg = (
    train.groupby("HubID")["OrderVolume"]
    .mean()
)

trend_ratio = (
    recent_avg / long_avg
).replace([np.inf, -np.inf], np.nan)

print(trend_ratio.describe())

print("\nMedian trend ratio:", trend_ratio.median())
print("Mean trend ratio:", trend_ratio.mean())

count    1115.000000
mean        1.045489
std         0.074456
min         0.746400
25%         1.009455
50%         1.041241
75%         1.079906
max         1.497291
Name: OrderVolume, dtype: float64

Median trend ratio: 1.0412409233878432
Mean trend ratio: 1.0454888892682306


In [58]:
# Compare last 56 days vs previous 56 days

last_56_start = train["Date"].max() - pd.Timedelta(days=55)
prev_56_start = train["Date"].max() - pd.Timedelta(days=111)

last56 = train[
    train["Date"] >= last_56_start
]

prev56 = train[
    (train["Date"] >= prev_56_start) &
    (train["Date"] < last_56_start)
]

last56_avg = last56.groupby("HubID")["OrderVolume"].mean()
prev56_avg = prev56.groupby("HubID")["OrderVolume"].mean()

momentum = (
    last56_avg / prev56_avg
).replace([np.inf, -np.inf], np.nan)

print(momentum.describe())
print("\nMedian momentum:", momentum.median())

count    1115.000000
mean        1.031567
std         0.046103
min         0.764324
25%         1.010535
50%         1.030259
75%         1.049269
max         1.691941
Name: OrderVolume, dtype: float64

Median momentum: 1.0302587960521359


In [59]:
# -------------------------------------------------------
# BUILD HUB-SPECIFIC MOMENTUM FROM HISTORY
# -------------------------------------------------------

def get_hub_momentum(history):
    
    max_date = history["Date"].max()
    
    last_56_start = max_date - pd.Timedelta(days=55)
    prev_56_start = max_date - pd.Timedelta(days=111)
    
    last56 = history[
        history["Date"] >= last_56_start
    ]
    
    prev56 = history[
        (history["Date"] >= prev_56_start) &
        (history["Date"] < last_56_start)
    ]
    
    last_avg = (
        last56.groupby("HubID")["OrderVolume"]
        .mean()
    )
    
    prev_avg = (
        prev56.groupby("HubID")["OrderVolume"]
        .mean()
    )
    
    momentum = (
        last_avg / prev_avg
    ).replace([np.inf, -np.inf], np.nan)
    
    # Avoid extreme ratios
    momentum = momentum.clip(
        lower=0.90,
        upper=1.15
    )
    
    return momentum


# -------------------------------------------------------
# BACKTEST DIFFERENT MOMENTUM STRENGTHS
# -------------------------------------------------------

base_pred, _ = statistical_forecast(
    hist_val,
    future_val.drop(columns=["OrderVolume"])
)

momentum_map = get_hub_momentum(hist_val)

hub_momentum = (
    future_val["HubID"]
    .map(momentum_map)
    .fillna(1.0)
    .values
)

actual = future_val["OrderVolume"].values

print("BASE SCORE")

base_score = np.sqrt(
    mean_squared_log_error(
        actual,
        np.clip(base_pred, 0, None)
    )
)

print("No adjustment:", base_score)

print("\nMOMENTUM TESTS")

strengths = [
    0.0,
    0.25,
    0.50,
    0.75,
    1.0
]

results = {}

for strength in strengths:
    
    # Apply partial momentum adjustment
    adjustment = (
        1 + strength * (hub_momentum - 1)
    )
    
    adjusted_pred = (
        base_pred.values * adjustment
    )
    
    adjusted_pred = np.clip(
        adjusted_pred,
        0,
        None
    )
    
    score = np.sqrt(
        mean_squared_log_error(
            actual,
            adjusted_pred
        )
    )
    
    results[strength] = score
    
    print(
        f"Strength {strength}: {score:.6f}"
    )

best_strength = min(
    results,
    key=results.get
)

print("\n" + "="*50)
print("BEST STRENGTH:", best_strength)
print("BEST SCORE:", results[best_strength])
print("="*50)

BASE SCORE
No adjustment: 0.2686154240232423

MOMENTUM TESTS
Strength 0.0: 0.268615
Strength 0.25: 0.268769
Strength 0.5: 0.269287
Strength 0.75: 0.270149
Strength 1.0: 0.271338

BEST STRENGTH: 0.0
BEST SCORE: 0.2686154240232423


In [66]:
from catboost import CatBoostRegressor
import numpy as np
import pandas as pd
from sklearn.metrics import mean_squared_log_error

# -------------------------------------------------------
# DATE FEATURES
# -------------------------------------------------------

def create_direct_features(df):
    
    df = df.copy()
    
    df["Year"] = df["Date"].dt.year
    df["Month"] = df["Date"].dt.month
    df["Day"] = df["Date"].dt.day
    df["DayOfYear"] = df["Date"].dt.dayofyear
    df["WeekOfYear"] = df["Date"].dt.isocalendar().week.astype(int)
    
    df["IsWeekend"] = (
        df["Weekday"] >= 5
    ).astype(int)
    
    df["Month_sin"] = np.sin(
        2 * np.pi * df["Month"] / 12
    )
    
    df["Month_cos"] = np.cos(
        2 * np.pi * df["Month"] / 12
    )
    
    return df


# -------------------------------------------------------
# BACKTEST DATA
# -------------------------------------------------------

direct_train = create_direct_features(hist_val)
direct_val = create_direct_features(
    future_val.drop(columns=["OrderVolume"])
)

# Merge metadata
meta_cols = [
    "HubID",
    "HubFormat",
    "AssortmentTier",
    "CompetitorDistance",
    "CompetitorOpenSinceMonth",
    "CompetitorOpenSinceYear",
    "LoyaltyProgram",
    "LoyaltyProgramSinceWeek",
    "LoyaltyProgramSinceYear",
    "LoyaltyProgramInterval"
]

for col in meta_cols:
    if col not in direct_train.columns:
        direct_train = direct_train.merge(
            metadata[meta_cols],
            on="HubID",
            how="left"
        )
        direct_val = direct_val.merge(
            metadata[meta_cols],
            on="HubID",
            how="left"
        )
        break


# -------------------------------------------------------
# FEATURES
# -------------------------------------------------------

DROP = [
    "Date",
    "OrderVolume",
    "Id"
]

DIRECT_FEATURES = [
    c for c in direct_train.columns
    if c not in DROP
]

DIRECT_CAT = [
    c for c in [
        "HubID",
        "HubFormat",
        "AssortmentTier",
        "LoyaltyProgram",
        "LoyaltyProgramInterval"
    ]
    if c in DIRECT_FEATURES
]

# CatBoost needs categorical columns as strings
for col in DIRECT_CAT:
    direct_train[col] = direct_train[col].fillna("Missing").astype(str)
    direct_val[col] = direct_val[col].fillna("Missing").astype(str)

# Numerical missing values
for col in DIRECT_FEATURES:
    if col not in DIRECT_CAT:
        direct_train[col] = direct_train[col].replace(
            [np.inf, -np.inf], np.nan
        ).fillna(-999)
        
        direct_val[col] = direct_val[col].replace(
            [np.inf, -np.inf], np.nan
        ).fillna(-999)


# -------------------------------------------------------
# TRAIN DIRECT MODEL
# -------------------------------------------------------

direct_model = CatBoostRegressor(
    iterations=600,
    depth=8,
    learning_rate=0.08,
    loss_function="RMSE",
    random_seed=42,
    verbose=100,
    thread_count=-1
)

direct_model.fit(
    direct_train[DIRECT_FEATURES],
    np.log1p(direct_train["OrderVolume"]),
    cat_features=DIRECT_CAT
)


# -------------------------------------------------------
# VALIDATE
# -------------------------------------------------------

direct_pred = np.expm1(
    direct_model.predict(
        direct_val[DIRECT_FEATURES]
    )
)

direct_pred = np.clip(
    direct_pred,
    0,
    None
)

# Closed hubs = zero
direct_pred[
    direct_val["IsOpen"].values == 0
] = 0


direct_score = np.sqrt(
    mean_squared_log_error(
        future_val["OrderVolume"],
        direct_pred
    )
)

print("\n" + "=" * 55)
print("DIRECT CATBOOST BACKTEST RMSLE:", direct_score)
print("STATISTICAL V2 BACKTEST      :", 0.268615)
print("=" * 55)

0:	learn: 3.0616553	total: 531ms	remaining: 5m 18s
100:	learn: 0.1028772	total: 37.7s	remaining: 3m 6s
200:	learn: 0.0820943	total: 1m 18s	remaining: 2m 36s
300:	learn: 0.0718988	total: 1m 57s	remaining: 1m 56s
400:	learn: 0.0667311	total: 2m 35s	remaining: 1m 17s
500:	learn: 0.0635530	total: 3m 13s	remaining: 38.3s
599:	learn: 0.0615839	total: 3m 53s	remaining: 0us

DIRECT CATBOOST BACKTEST RMSLE: 0.06280981625034524
STATISTICAL V2 BACKTEST      : 0.268615


In [68]:
# ============================================================
# FINAL DIRECT CATBOOST MODEL - CORRECTED
# ============================================================

full_train = create_direct_features(train.copy())
full_test = create_direct_features(test.copy())

# Merge metadata
if "HubFormat" not in full_train.columns:
    full_train = full_train.merge(
        metadata[meta_cols],
        on="HubID",
        how="left"
    )

if "HubFormat" not in full_test.columns:
    full_test = full_test.merge(
        metadata[meta_cols],
        on="HubID",
        how="left"
    )


# ============================================================
# IMPORTANT: REMOVE COLUMNS NOT AVAILABLE IN TEST
# ============================================================

DROP = [
    "Date",
    "OrderVolume",
    "Id",
    "AppSessions"       # <-- IMPORTANT FIX
]

FINAL_FEATURES = [
    c for c in full_train.columns
    if c not in DROP
    and c in full_test.columns
]

FINAL_CAT = [
    c for c in [
        "HubID",
        "HubFormat",
        "AssortmentTier",
        "LoyaltyProgram",
        "LoyaltyProgramInterval"
    ]
    if c in FINAL_FEATURES
]

print("Features:", len(FINAL_FEATURES))
print("Categorical:", FINAL_CAT)

print("\nMissing from test:")
print(set(FINAL_FEATURES) - set(full_test.columns))


# ============================================================
# PREPARE CATEGORICAL FEATURES
# ============================================================

for col in FINAL_CAT:
    
    full_train[col] = (
        full_train[col]
        .fillna("Missing")
        .astype(str)
    )
    
    full_test[col] = (
        full_test[col]
        .fillna("Missing")
        .astype(str)
    )


# ============================================================
# PREPARE NUMERICAL FEATURES
# ============================================================

for col in FINAL_FEATURES:
    
    if col not in FINAL_CAT:
        
        full_train[col] = (
            full_train[col]
            .replace([np.inf, -np.inf], np.nan)
            .fillna(-999)
        )
        
        full_test[col] = (
            full_test[col]
            .replace([np.inf, -np.inf], np.nan)
            .fillna(-999)
        )


# ============================================================
# TRAIN FINAL MODEL
# ============================================================

final_model = CatBoostRegressor(
    iterations=750,
    depth=8,
    learning_rate=0.08,
    loss_function="RMSE",
    random_seed=42,
    verbose=100,
    thread_count=-1
)

final_model.fit(
    full_train[FINAL_FEATURES],
    np.log1p(full_train["OrderVolume"]),
    cat_features=FINAL_CAT
)

print("\n" + "=" * 50)
print("FINAL MODEL TRAINING COMPLETE!")
print("=" * 50)

Features: 23
Categorical: ['HubID', 'HubFormat', 'AssortmentTier', 'LoyaltyProgram', 'LoyaltyProgramInterval']

Missing from test:
set()
0:	learn: 3.0604053	total: 423ms	remaining: 5m 16s
100:	learn: 0.1770440	total: 53.1s	remaining: 5m 41s
200:	learn: 0.1592374	total: 1m 53s	remaining: 5m 10s
300:	learn: 0.1506329	total: 2m 53s	remaining: 4m 18s
400:	learn: 0.1450330	total: 3m 50s	remaining: 3m 20s
500:	learn: 0.1412736	total: 4m 52s	remaining: 2m 25s
600:	learn: 0.1380643	total: 5m 57s	remaining: 1m 28s
700:	learn: 0.1355006	total: 6m 58s	remaining: 29.2s
749:	learn: 0.1343454	total: 7m 27s	remaining: 0us

FINAL MODEL TRAINING COMPLETE!


In [69]:
# ============================================================
# FINAL PREDICTION + SUBMISSION
# ============================================================

final_pred = np.expm1(
    final_model.predict(full_test[FINAL_FEATURES])
)

final_pred = np.clip(final_pred, 0, None)

# Closed hubs must have zero orders
final_pred[full_test["IsOpen"].values == 0] = 0


# Create submission
final_submission = pd.DataFrame({
    "Id": test["Id"],
    "OrderVolume": final_pred
})


# Checks
print("Submission shape:", final_submission.shape)

print("\nMissing predictions:")
print(final_submission.isnull().sum())

print("\nPrediction statistics:")
print(final_submission["OrderVolume"].describe())

display(final_submission.head())


# Save
final_submission.to_csv(
    "/kaggle/working/submission_direct_catboost.csv",
    index=False
)

print("\n" + "=" * 60)
print("FILE SAVED SUCCESSFULLY!")
print("submission_direct_catboost.csv")
print("=" * 60)

Submission shape: (46830, 2)

Missing predictions:
Id             0
OrderVolume    0
dtype: int64

Prediction statistics:
count    46830.000000
mean      6011.533374
std       3562.305586
min          0.000000
25%       4177.245435
50%       5995.842193
75%       8071.058928
max      29281.757790
Name: OrderVolume, dtype: float64


,Id,OrderVolume
0,1,4566.299238
1,2,2804.758707
2,3,4014.921147
3,4,10458.596924
4,5,2011.147912



FILE SAVED SUCCESSFULLY!
submission_direct_catboost.csv


In [70]:
# ============================================================
# PROPER VALIDATION - NO APPSessions
# Same setup as final competition model
# ============================================================

VAL_START = pd.Timestamp("2014-06-20")
VAL_END = pd.Timestamp("2014-07-31")

bt_train = train[train["Date"] < VAL_START].copy()

bt_val = train[
    (train["Date"] >= VAL_START) &
    (train["Date"] <= VAL_END)
].copy()

# Create date features
bt_train_feat = create_direct_features(bt_train.copy())
bt_val_feat = create_direct_features(bt_val.copy())

# Merge metadata
if "HubFormat" not in bt_train_feat.columns:
    bt_train_feat = bt_train_feat.merge(
        metadata[meta_cols],
        on="HubID",
        how="left"
    )

if "HubFormat" not in bt_val_feat.columns:
    bt_val_feat = bt_val_feat.merge(
        metadata[meta_cols],
        on="HubID",
        how="left"
    )

DROP_BT = [
    "Date",
    "OrderVolume",
    "Id",
    "AppSessions"
]

BT_FEATURES = [
    c for c in bt_train_feat.columns
    if c not in DROP_BT
    and c in bt_val_feat.columns
]

BT_CAT = [
    c for c in [
        "HubID",
        "HubFormat",
        "AssortmentTier",
        "LoyaltyProgram",
        "LoyaltyProgramInterval"
    ]
    if c in BT_FEATURES
]

print("Features:", len(BT_FEATURES))
print("Train shape:", bt_train_feat.shape)
print("Validation shape:", bt_val_feat.shape)

# Prepare categorical
for col in BT_CAT:
    bt_train_feat[col] = bt_train_feat[col].fillna("Missing").astype(str)
    bt_val_feat[col] = bt_val_feat[col].fillna("Missing").astype(str)

# Prepare numeric
for col in BT_FEATURES:
    if col not in BT_CAT:
        bt_train_feat[col] = bt_train_feat[col].replace(
            [np.inf, -np.inf], np.nan
        ).fillna(-999)

        bt_val_feat[col] = bt_val_feat[col].replace(
            [np.inf, -np.inf], np.nan
        ).fillna(-999)

Features: 23
Train shape: (596524, 26)
Validation shape: (41250, 26)


In [71]:
validation_model = CatBoostRegressor(
    iterations=750,
    depth=8,
    learning_rate=0.08,
    loss_function="RMSE",
    random_seed=42,
    verbose=100,
    thread_count=-1
)

validation_model.fit(
    bt_train_feat[BT_FEATURES],
    np.log1p(bt_train_feat["OrderVolume"]),
    cat_features=BT_CAT
)

bt_pred = np.expm1(
    validation_model.predict(bt_val_feat[BT_FEATURES])
)

bt_pred = np.clip(bt_pred, 0, None)

bt_pred[bt_val_feat["IsOpen"].values == 0] = 0

bt_score = np.sqrt(
    mean_squared_log_error(
        bt_val_feat["OrderVolume"],
        bt_pred
    )
)

print("\nCURRENT MODEL PROPER BACKTEST:", bt_score)

0:	learn: 3.0640596	total: 541ms	remaining: 6m 44s
100:	learn: 0.1783705	total: 34.5s	remaining: 3m 41s
200:	learn: 0.1614287	total: 1m 10s	remaining: 3m 13s
300:	learn: 0.1521616	total: 1m 50s	remaining: 2m 44s
400:	learn: 0.1469670	total: 2m 28s	remaining: 2m 9s
500:	learn: 0.1429513	total: 3m 5s	remaining: 1m 32s
600:	learn: 0.1398869	total: 3m 42s	remaining: 55.1s
700:	learn: 0.1372271	total: 4m 19s	remaining: 18.2s
749:	learn: 0.1358068	total: 4m 38s	remaining: 0us

CURRENT MODEL PROPER BACKTEST: 0.19368760256758605


In [72]:
def create_enhanced_features(df):
    
    df = df.copy()
    
    # Basic calendar
    df["Year"] = df["Date"].dt.year
    df["Month"] = df["Date"].dt.month
    df["Day"] = df["Date"].dt.day
    df["DayOfYear"] = df["Date"].dt.dayofyear
    df["WeekOfYear"] = df["Date"].dt.isocalendar().week.astype(int)
    df["Quarter"] = df["Date"].dt.quarter
    
    # Trend
    df["DaysSinceStart"] = (
        df["Date"] - pd.Timestamp("2013-01-01")
    ).dt.days
    
    # Weekend
    df["IsWeekend"] = (
        df["Weekday"] >= 5
    ).astype(int)
    
    # Month position
    df["WeekOfMonth"] = (
        (df["Day"] - 1) // 7 + 1
    ).astype(int)
    
    # Cyclic yearly seasonality
    df["DayOfYear_sin"] = np.sin(
        2 * np.pi * df["DayOfYear"] / 365.25
    )
    
    df["DayOfYear_cos"] = np.cos(
        2 * np.pi * df["DayOfYear"] / 365.25
    )
    
    # Weekly cyclic
    df["Weekday_sin"] = np.sin(
        2 * np.pi * df["Weekday"] / 7
    )
    
    df["Weekday_cos"] = np.cos(
        2 * np.pi * df["Weekday"] / 7
    )
    
    # Month cyclic
    df["Month_sin"] = np.sin(
        2 * np.pi * df["Month"] / 12
    )
    
    df["Month_cos"] = np.cos(
        2 * np.pi * df["Month"] / 12
    )
    
    # Operational interactions
    df["PromoWeekend"] = (
        df["PromoActive"] * df["IsWeekend"]
    )
    
    df["PromoHoliday"] = (
        df["PromoActive"] * (df["RegionalHoliday"] > 0).astype(int)
    )
    
    df["HolidayWeekend"] = (
        (df["RegionalHoliday"] > 0).astype(int)
        * df["IsWeekend"]
    )
    
    df["SchoolWeekend"] = (
        df["SchoolClosureFlag"] * df["IsWeekend"]
    )
    
    return df

In [73]:
def add_metadata_features(df):
    
    df = df.copy()
    
    if "HubFormat" not in df.columns:
        df = df.merge(
            metadata[meta_cols],
            on="HubID",
            how="left"
        )
    
    # Competitor age
    if (
        "CompetitorOpenSinceYear" in df.columns and
        "CompetitorOpenSinceMonth" in df.columns
    ):
        
        competitor_year = (
            df["CompetitorOpenSinceYear"]
            .fillna(df["Date"].dt.year)
        )
        
        competitor_month = (
            df["CompetitorOpenSinceMonth"]
            .fillna(1)
        )
        
        df["CompetitorAgeMonths"] = (
            (df["Date"].dt.year - competitor_year) * 12
            + df["Date"].dt.month
            - competitor_month
        )
        
        df["CompetitorAgeMonths"] = (
            df["CompetitorAgeMonths"]
            .clip(lower=0)
        )
    
    
    # Loyalty age
    if "LoyaltyProgramSinceYear" in df.columns:
        
        loyalty_year = df[
            "LoyaltyProgramSinceYear"
        ].fillna(df["Date"].dt.year)
        
        loyalty_week = df[
            "LoyaltyProgramSinceWeek"
        ].fillna(1)
        
        current_week = df[
            "WeekOfYear"
        ]
        
        df["LoyaltyAgeWeeks"] = (
            (df["Date"].dt.year - loyalty_year) * 52
            + current_week
            - loyalty_week
        )
        
        df["LoyaltyAgeWeeks"] = (
            df["LoyaltyAgeWeeks"]
            .clip(lower=0)
        )
    
    return df

In [74]:
# Build enhanced data

enh_train = create_enhanced_features(bt_train.copy())
enh_val = create_enhanced_features(bt_val.copy())

enh_train = add_metadata_features(enh_train)
enh_val = add_metadata_features(enh_val)


DROP_ENH = [
    "Date",
    "OrderVolume",
    "Id",
    "AppSessions"
]

ENH_FEATURES = [
    c for c in enh_train.columns
    if c not in DROP_ENH
    and c in enh_val.columns
]

ENH_CAT = [
    c for c in [
        "HubID",
        "HubFormat",
        "AssortmentTier",
        "LoyaltyProgram",
        "LoyaltyProgramInterval"
    ]
    if c in ENH_FEATURES
]


print("Enhanced features:", len(ENH_FEATURES))


# Prepare categories
for col in ENH_CAT:
    
    enh_train[col] = enh_train[col].fillna("Missing").astype(str)
    enh_val[col] = enh_val[col].fillna("Missing").astype(str)


# Prepare numerics
for col in ENH_FEATURES:
    
    if col not in ENH_CAT:
        
        enh_train[col] = (
            enh_train[col]
            .replace([np.inf, -np.inf], np.nan)
            .fillna(-999)
        )
        
        enh_val[col] = (
            enh_val[col]
            .replace([np.inf, -np.inf], np.nan)
            .fillna(-999)
        )


# Train

enh_model = CatBoostRegressor(
    iterations=900,
    depth=9,
    learning_rate=0.06,
    l2_leaf_reg=5,
    loss_function="RMSE",
    random_seed=42,
    verbose=100,
    thread_count=-1
)

enh_model.fit(
    enh_train[ENH_FEATURES],
    np.log1p(enh_train["OrderVolume"]),
    cat_features=ENH_CAT
)


# Predict

enh_pred = np.expm1(
    enh_model.predict(
        enh_val[ENH_FEATURES]
    )
)

enh_pred = np.clip(enh_pred, 0, None)

enh_pred[
    enh_val["IsOpen"].values == 0
] = 0


enh_score = np.sqrt(
    mean_squared_log_error(
        enh_val["OrderVolume"],
        enh_pred
    )
)

print("\n" + "=" * 55)
print("CURRENT BACKTEST :", 0.1936876)
print("ENHANCED BACKTEST:", enh_score)
print("=" * 55)

Enhanced features: 36
0:	learn: 3.1296165	total: 698ms	remaining: 10m 27s
100:	learn: 0.1826182	total: 41s	remaining: 5m 24s
200:	learn: 0.1604204	total: 1m 31s	remaining: 5m 18s
300:	learn: 0.1502882	total: 2m 18s	remaining: 4m 35s
400:	learn: 0.1443152	total: 3m 2s	remaining: 3m 47s
500:	learn: 0.1404093	total: 3m 47s	remaining: 3m 1s
600:	learn: 0.1366769	total: 4m 37s	remaining: 2m 17s
700:	learn: 0.1342428	total: 5m 35s	remaining: 1m 35s
800:	learn: 0.1319634	total: 6m 29s	remaining: 48.2s
899:	learn: 0.1299374	total: 7m 25s	remaining: 0us

CURRENT BACKTEST : 0.1936876
ENHANCED BACKTEST: 0.1888019437917548


In [75]:
# ============================================================
# BLEND CURRENT + ENHANCED MODELS
# ============================================================

y_true = bt_val_feat["OrderVolume"].values

print("MODEL SCORES")
print("-" * 40)

for w in np.arange(0, 1.05, 0.05):

    # w = weight for enhanced model
    blend_pred = (
        (1 - w) * bt_pred
        + w * enh_pred
    )

    blend_pred = np.clip(blend_pred, 0, None)

    score = np.sqrt(
        mean_squared_log_error(
            y_true,
            blend_pred
        )
    )

    print(f"Enhanced weight {w:.2f}: {score:.6f}")

MODEL SCORES
----------------------------------------
Enhanced weight 0.00: 0.193688
Enhanced weight 0.05: 0.193219
Enhanced weight 0.10: 0.192770
Enhanced weight 0.15: 0.192342
Enhanced weight 0.20: 0.191936
Enhanced weight 0.25: 0.191552
Enhanced weight 0.30: 0.191191
Enhanced weight 0.35: 0.190852
Enhanced weight 0.40: 0.190536
Enhanced weight 0.45: 0.190245
Enhanced weight 0.50: 0.189978
Enhanced weight 0.55: 0.189736
Enhanced weight 0.60: 0.189520
Enhanced weight 0.65: 0.189330
Enhanced weight 0.70: 0.189167
Enhanced weight 0.75: 0.189032
Enhanced weight 0.80: 0.188925
Enhanced weight 0.85: 0.188848
Enhanced weight 0.90: 0.188801
Enhanced weight 0.95: 0.188785
Enhanced weight 1.00: 0.188802


In [77]:
# ============================================================
# LOG-SPACE / GEOMETRIC BLEND
# ============================================================

print("LOG-SPACE BLEND SCORES")
print("-" * 40)

results = []

for w in np.arange(0, 1.05, 0.05):

    log_blend = (
        (1 - w) * np.log1p(bt_pred)
        + w * np.log1p(enh_pred)
    )

    blend_pred = np.expm1(log_blend)
    blend_pred = np.clip(blend_pred, 0, None)

    score = np.sqrt(
        mean_squared_log_error(y_true, blend_pred)
    )

    results.append((w, score))

    print(f"Enhanced weight {w:.2f}: {score:.6f}")

best_w, best_score = min(results, key=lambda x: x[1])

print("\nBEST WEIGHT:", best_w)
print("BEST SCORE :", best_score)

LOG-SPACE BLEND SCORES
----------------------------------------
Enhanced weight 0.00: 0.193688
Enhanced weight 0.05: 0.193205
Enhanced weight 0.10: 0.192746
Enhanced weight 0.15: 0.192312
Enhanced weight 0.20: 0.191903
Enhanced weight 0.25: 0.191518
Enhanced weight 0.30: 0.191158
Enhanced weight 0.35: 0.190823
Enhanced weight 0.40: 0.190514
Enhanced weight 0.45: 0.190230
Enhanced weight 0.50: 0.189971
Enhanced weight 0.55: 0.189738
Enhanced weight 0.60: 0.189531
Enhanced weight 0.65: 0.189349
Enhanced weight 0.70: 0.189193
Enhanced weight 0.75: 0.189063
Enhanced weight 0.80: 0.188959
Enhanced weight 0.85: 0.188880
Enhanced weight 0.90: 0.188828
Enhanced weight 0.95: 0.188802
Enhanced weight 1.00: 0.188802

BEST WEIGHT: 1.0
BEST SCORE : 0.1888019437917548


In [78]:
# ============================================================
# HUB HISTORICAL TARGET FEATURES
# ============================================================

def add_historical_hub_features(train_df, future_df):
    
    train_df = train_df.copy()
    future_df = future_df.copy()
    
    # --------------------------------------------------------
    # HUB GLOBAL STATISTICS
    # --------------------------------------------------------
    
    hub_stats = (
        train_df.groupby("HubID")["OrderVolume"]
        .agg([
            "mean",
            "median",
            "std",
            "min",
            "max"
        ])
        .reset_index()
    )
    
    hub_stats.columns = [
        "HubID",
        "HubHistoricalMean",
        "HubHistoricalMedian",
        "HubHistoricalStd",
        "HubHistoricalMin",
        "HubHistoricalMax"
    ]
    
    future_df = future_df.merge(
        hub_stats,
        on="HubID",
        how="left"
    )
    
    
    # --------------------------------------------------------
    # HUB + WEEKDAY AVERAGE
    # --------------------------------------------------------
    
    hub_weekday = (
        train_df.groupby(
            ["HubID", "Weekday"]
        )["OrderVolume"]
        .agg(["mean", "median"])
        .reset_index()
    )
    
    hub_weekday.columns = [
        "HubID",
        "Weekday",
        "HubWeekdayMean",
        "HubWeekdayMedian"
    ]
    
    future_df = future_df.merge(
        hub_weekday,
        on=["HubID", "Weekday"],
        how="left"
    )
    
    
    # --------------------------------------------------------
    # HUB + PROMO AVERAGE
    # --------------------------------------------------------
    
    hub_promo = (
        train_df.groupby(
            ["HubID", "PromoActive"]
        )["OrderVolume"]
        .mean()
        .reset_index()
        .rename(
            columns={
                "OrderVolume": "HubPromoMean"
            }
        )
    )
    
    future_df = future_df.merge(
        hub_promo,
        on=["HubID", "PromoActive"],
        how="left"
    )
    
    
    # --------------------------------------------------------
    # HUB + HOLIDAY AVERAGE
    # --------------------------------------------------------
    
    hub_holiday = (
        train_df.groupby(
            ["HubID", "RegionalHoliday"]
        )["OrderVolume"]
        .mean()
        .reset_index()
        .rename(
            columns={
                "OrderVolume": "HubHolidayMean"
            }
        )
    )
    
    future_df = future_df.merge(
        hub_holiday,
        on=["HubID", "RegionalHoliday"],
        how="left"
    )
    
    
    # Fill missing target features
    
    target_features = [
        "HubHistoricalMean",
        "HubHistoricalMedian",
        "HubHistoricalStd",
        "HubHistoricalMin",
        "HubHistoricalMax",
        "HubWeekdayMean",
        "HubWeekdayMedian",
        "HubPromoMean",
        "HubHolidayMean"
    ]
    
    for col in target_features:
        
        if col in future_df.columns:
            
            future_df[col] = future_df[col].fillna(
                train_df["OrderVolume"].mean()
            )
    
    return future_df

In [79]:
# ============================================================
# V5 VALIDATION DATA
# Enhanced Features + Historical Hub Features
# ============================================================

v5_train = create_enhanced_features(bt_train.copy())
v5_val = create_enhanced_features(bt_val.copy())

v5_train = add_metadata_features(v5_train)
v5_val = add_metadata_features(v5_val)


# Add historical features
# Validation gets stats ONLY from bt_train

v5_val = add_historical_hub_features(
    bt_train,
    v5_val
)

# For training rows, we need stats as features too
# Use aggregate stats from training itself

v5_train = add_historical_hub_features(
    bt_train,
    v5_train
)


DROP_V5 = [
    "Date",
    "OrderVolume",
    "Id",
    "AppSessions"
]

V5_FEATURES = [
    c for c in v5_train.columns
    if c not in DROP_V5
    and c in v5_val.columns
]

V5_CAT = [
    c for c in [
        "HubID",
        "HubFormat",
        "AssortmentTier",
        "LoyaltyProgram",
        "LoyaltyProgramInterval"
    ]
    if c in V5_FEATURES
]


print("V5 Features:", len(V5_FEATURES))
print("Categorical:", V5_CAT)

V5 Features: 45
Categorical: ['HubID', 'HubFormat', 'AssortmentTier', 'LoyaltyProgram', 'LoyaltyProgramInterval']


In [80]:
# ============================================================
# PREPARE V5 DATA
# ============================================================

for col in V5_CAT:
    
    v5_train[col] = (
        v5_train[col]
        .fillna("Missing")
        .astype(str)
    )
    
    v5_val[col] = (
        v5_val[col]
        .fillna("Missing")
        .astype(str)
    )


for col in V5_FEATURES:
    
    if col not in V5_CAT:
        
        v5_train[col] = (
            v5_train[col]
            .replace([np.inf, -np.inf], np.nan)
            .fillna(-999)
        )
        
        v5_val[col] = (
            v5_val[col]
            .replace([np.inf, -np.inf], np.nan)
            .fillna(-999)
        )

In [81]:
# ============================================================
# TRAIN V5
# ============================================================

v5_model = CatBoostRegressor(
    iterations=900,
    depth=9,
    learning_rate=0.06,
    l2_leaf_reg=5,
    loss_function="RMSE",
    random_seed=42,
    verbose=100,
    thread_count=-1
)

v5_model.fit(
    v5_train[V5_FEATURES],
    np.log1p(v5_train["OrderVolume"]),
    cat_features=V5_CAT
)


v5_pred = np.expm1(
    v5_model.predict(
        v5_val[V5_FEATURES]
    )
)

v5_pred = np.clip(v5_pred, 0, None)

v5_pred[
    v5_val["IsOpen"].values == 0
] = 0


v5_score = np.sqrt(
    mean_squared_log_error(
        v5_val["OrderVolume"],
        v5_pred
    )
)

print("\n" + "=" * 60)
print("V3 BACKTEST:", 0.193688)
print("V4 BACKTEST:", 0.188802)
print("V5 BACKTEST:", v5_score)
print("=" * 60)

0:	learn: 3.1284591	total: 556ms	remaining: 8m 19s
100:	learn: 0.1361562	total: 46.9s	remaining: 6m 11s
200:	learn: 0.1250810	total: 1m 34s	remaining: 5m 29s
300:	learn: 0.1206100	total: 2m 22s	remaining: 4m 44s
400:	learn: 0.1176426	total: 3m 9s	remaining: 3m 56s
500:	learn: 0.1151548	total: 3m 57s	remaining: 3m 8s
600:	learn: 0.1130811	total: 4m 43s	remaining: 2m 21s
700:	learn: 0.1112331	total: 5m 30s	remaining: 1m 33s
800:	learn: 0.1096066	total: 6m 16s	remaining: 46.6s
899:	learn: 0.1079758	total: 7m 3s	remaining: 0us

V3 BACKTEST: 0.193688
V4 BACKTEST: 0.188802
V5 BACKTEST: 0.18633603312447766


In [82]:
# ============================================================
# RECENT HUB HISTORY FEATURES - LEAKAGE SAFE
# ============================================================

def add_recent_history_features(history_df, future_df):
    
    history_df = history_df.copy()
    future_df = future_df.copy()
    
    latest_date = history_df["Date"].max()
    
    windows = [7, 14, 28, 56, 112]
    
    feature_frames = []
    
    for window in windows:
        
        cutoff = latest_date - pd.Timedelta(days=window - 1)
        
        recent = history_df[
            history_df["Date"] >= cutoff
        ]
        
        stats = (
            recent.groupby("HubID")["OrderVolume"]
            .agg(["mean", "median", "std", "min", "max"])
            .reset_index()
        )
        
        stats.columns = [
            "HubID",
            f"Recent{window}Mean",
            f"Recent{window}Median",
            f"Recent{window}Std",
            f"Recent{window}Min",
            f"Recent{window}Max"
        ]
        
        future_df = future_df.merge(
            stats,
            on="HubID",
            how="left"
        )
    
    
    # Same weekday averages from recent 56 days
    
    recent56 = history_df[
        history_df["Date"] >= latest_date - pd.Timedelta(days=55)
    ]
    
    weekday_stats = (
        recent56.groupby(["HubID", "Weekday"])["OrderVolume"]
        .agg(["mean", "median"])
        .reset_index()
    )
    
    weekday_stats.columns = [
        "HubID",
        "Weekday",
        "RecentWeekdayMean",
        "RecentWeekdayMedian"
    ]
    
    future_df = future_df.merge(
        weekday_stats,
        on=["HubID", "Weekday"],
        how="left"
    )
    
    
    # Trend ratios
    
    if "Recent28Mean" in future_df.columns:
        
        future_df["Trend7vs28"] = (
            future_df["Recent7Mean"] /
            (future_df["Recent28Mean"] + 1)
        )
        
        future_df["Trend14vs56"] = (
            future_df["Recent14Mean"] /
            (future_df["Recent56Mean"] + 1)
        )
        
        future_df["Trend28vs112"] = (
            future_df["Recent28Mean"] /
            (future_df["Recent112Mean"] + 1)
        )
    
    
    return future_df

In [83]:
# ============================================================
# V6 - ENHANCED + RECENT HISTORY FEATURES
# ============================================================

v6_train = create_enhanced_features(bt_train.copy())
v6_val = create_enhanced_features(bt_val.copy())

v6_train = add_metadata_features(v6_train)
v6_val = add_metadata_features(v6_val)


# Add recent historical features
# Both use ONLY bt_train history for validation experiment

v6_val = add_recent_history_features(
    bt_train,
    v6_val
)

v6_train = add_recent_history_features(
    bt_train,
    v6_train
)


DROP_V6 = [
    "Date",
    "OrderVolume",
    "Id",
    "AppSessions"
]

V6_FEATURES = [
    c for c in v6_train.columns
    if c not in DROP_V6
    and c in v6_val.columns
]

V6_CAT = [
    c for c in [
        "HubID",
        "HubFormat",
        "AssortmentTier",
        "LoyaltyProgram",
        "LoyaltyProgramInterval"
    ]
    if c in V6_FEATURES
]

print("V6 Features:", len(V6_FEATURES))

V6 Features: 66


In [84]:
for col in V6_CAT:
    v6_train[col] = v6_train[col].fillna("Missing").astype(str)
    v6_val[col] = v6_val[col].fillna("Missing").astype(str)

for col in V6_FEATURES:
    if col not in V6_CAT:
        v6_train[col] = (
            v6_train[col]
            .replace([np.inf, -np.inf], np.nan)
            .fillna(-999)
        )
        
        v6_val[col] = (
            v6_val[col]
            .replace([np.inf, -np.inf], np.nan)
            .fillna(-999)
        )


v6_model = CatBoostRegressor(
    iterations=900,
    depth=9,
    learning_rate=0.06,
    l2_leaf_reg=5,
    loss_function="RMSE",
    random_seed=42,
    verbose=100,
    thread_count=-1
)

v6_model.fit(
    v6_train[V6_FEATURES],
    np.log1p(v6_train["OrderVolume"]),
    cat_features=V6_CAT
)

v6_pred = np.expm1(
    v6_model.predict(v6_val[V6_FEATURES])
)

v6_pred = np.clip(v6_pred, 0, None)

v6_pred[v6_val["IsOpen"].values == 0] = 0

v6_score = np.sqrt(
    mean_squared_log_error(
        v6_val["OrderVolume"],
        v6_pred
    )
)

print("\n" + "=" * 55)
print("V4:", 0.188802)
print("V5:", 0.186336)
print("V6:", v6_score)
print("=" * 55)

0:	learn: 3.1286531	total: 636ms	remaining: 9m 31s
100:	learn: 0.1468899	total: 53.7s	remaining: 7m 4s
200:	learn: 0.1302651	total: 1m 47s	remaining: 6m 12s
300:	learn: 0.1235185	total: 2m 40s	remaining: 5m 19s
400:	learn: 0.1191150	total: 3m 34s	remaining: 4m 27s
500:	learn: 0.1157803	total: 4m 28s	remaining: 3m 33s
600:	learn: 0.1130298	total: 5m 21s	remaining: 2m 39s
700:	learn: 0.1107502	total: 6m 14s	remaining: 1m 46s
800:	learn: 0.1088431	total: 7m 7s	remaining: 52.8s
899:	learn: 0.1070057	total: 8m	remaining: 0us

V4: 0.188802
V5: 0.186336
V6: 0.18966110582721507


In [85]:
# ============================================================
# V5 FINAL SUBMISSION
# Enhanced Features + Historical Hub Statistics
# ============================================================

# Start with enhanced calendar features
v5_full_train = create_enhanced_features(train.copy())
v5_full_test = create_enhanced_features(test.copy())

# Add metadata features
v5_full_train = add_metadata_features(v5_full_train)
v5_full_test = add_metadata_features(v5_full_test)

print("Train shape:", v5_full_train.shape)
print("Test shape :", v5_full_test.shape)

Train shape: (970379, 39)
Test shape : (46830, 38)


In [86]:
# ============================================================
# ADD HISTORICAL TARGET FEATURES
# ============================================================

# Test gets statistics from full training history
v5_full_test = add_historical_hub_features(
    train,
    v5_full_test
)

# Training gets hub statistics
v5_full_train = add_historical_hub_features(
    train,
    v5_full_train
)

print("Historical features added.")

print("\nTrain columns:", len(v5_full_train.columns))
print("Test columns :", len(v5_full_test.columns))

Historical features added.

Train columns: 48
Test columns : 47


In [87]:
# ============================================================
# DEFINE FINAL FEATURES
# ============================================================

DROP_FINAL_V5 = [
    "Date",
    "OrderVolume",
    "Id",
    "AppSessions"
]

FINAL_V5_FEATURES = [
    c for c in v5_full_train.columns
    if c not in DROP_FINAL_V5
    and c in v5_full_test.columns
]

FINAL_V5_CAT = [
    c for c in [
        "HubID",
        "HubFormat",
        "AssortmentTier",
        "LoyaltyProgram",
        "LoyaltyProgramInterval"
    ]
    if c in FINAL_V5_FEATURES
]

print("Number of features:", len(FINAL_V5_FEATURES))
print("Categorical:", FINAL_V5_CAT)

print("\nFeatures missing from test:")
print(set(FINAL_V5_FEATURES) - set(v5_full_test.columns))

Number of features: 45
Categorical: ['HubID', 'HubFormat', 'AssortmentTier', 'LoyaltyProgram', 'LoyaltyProgramInterval']

Features missing from test:
set()


In [88]:
# ============================================================
# DATA PREPARATION
# ============================================================

# Categorical features
for col in FINAL_V5_CAT:

    v5_full_train[col] = (
        v5_full_train[col]
        .fillna("Missing")
        .astype(str)
    )

    v5_full_test[col] = (
        v5_full_test[col]
        .fillna("Missing")
        .astype(str)
    )


# Numerical features
for col in FINAL_V5_FEATURES:

    if col not in FINAL_V5_CAT:

        v5_full_train[col] = (
            v5_full_train[col]
            .replace([np.inf, -np.inf], np.nan)
            .fillna(-999)
        )

        v5_full_test[col] = (
            v5_full_test[col]
            .replace([np.inf, -np.inf], np.nan)
            .fillna(-999)
        )


print("Data preparation complete.")

print(
    "Train feature shape:",
    v5_full_train[FINAL_V5_FEATURES].shape
)

print(
    "Test feature shape:",
    v5_full_test[FINAL_V5_FEATURES].shape
)

Data preparation complete.
Train feature shape: (970379, 45)
Test feature shape: (46830, 45)


In [89]:
# ============================================================
# FINAL V5 CATBOOST MODEL
# ============================================================

final_v5_model = CatBoostRegressor(
    iterations=900,
    depth=9,
    learning_rate=0.06,
    l2_leaf_reg=5,
    loss_function="RMSE",
    random_seed=42,
    verbose=100,
    thread_count=-1
)

final_v5_model.fit(
    v5_full_train[FINAL_V5_FEATURES],
    np.log1p(v5_full_train["OrderVolume"]),
    cat_features=FINAL_V5_CAT
)

print("\n" + "=" * 60)
print("FINAL V5 MODEL TRAINING COMPLETE!")
print("=" * 60)

0:	learn: 3.1250582	total: 470ms	remaining: 7m 2s
100:	learn: 0.1386712	total: 1m 14s	remaining: 9m 48s
200:	learn: 0.1276170	total: 2m 31s	remaining: 8m 47s
300:	learn: 0.1227936	total: 3m 50s	remaining: 7m 38s
400:	learn: 0.1197003	total: 5m 20s	remaining: 6m 39s
500:	learn: 0.1172789	total: 6m 39s	remaining: 5m 17s
600:	learn: 0.1154371	total: 7m 57s	remaining: 3m 57s
700:	learn: 0.1139344	total: 9m 13s	remaining: 2m 37s
800:	learn: 0.1125833	total: 10m 30s	remaining: 1m 17s
899:	learn: 0.1113123	total: 11m 43s	remaining: 0us

FINAL V5 MODEL TRAINING COMPLETE!


In [90]:
# ============================================================
# FINAL V5 PREDICTIONS
# ============================================================

v5_final_pred = np.expm1(
    final_v5_model.predict(
        v5_full_test[FINAL_V5_FEATURES]
    )
)

v5_final_pred = np.clip(v5_final_pred, 0, None)

# Closed hubs = zero orders
v5_final_pred[
    v5_full_test["IsOpen"].values == 0
] = 0

print("Prediction statistics:")
print(pd.Series(v5_final_pred).describe())

Prediction statistics:
count    46830.000000
mean      5980.814539
std       3619.327320
min          0.000000
25%       4180.268728
50%       5910.465767
75%       7937.820181
max      28783.330617
dtype: float64


In [91]:
# ============================================================
# CREATE V5 SUBMISSION
# ============================================================

submission_v5 = pd.DataFrame({
    "Id": test["Id"],
    "OrderVolume": v5_final_pred
})

print("Submission shape:", submission_v5.shape)

print("\nMissing values:")
print(submission_v5.isnull().sum())

display(submission_v5.head())

submission_v5.to_csv(
    "/kaggle/working/submission_v5_hubstats.csv",
    index=False
)

print("\n" + "=" * 60)
print("SUBMISSION SAVED SUCCESSFULLY")
print("File: submission_v5_hubstats.csv")
print("=" * 60)

Submission shape: (46830, 2)

Missing values:
Id             0
OrderVolume    0
dtype: int64


,Id,OrderVolume
0,1,4776.603131
1,2,2786.469390
2,3,4156.935986
3,4,9600.432800
4,5,2041.474217



SUBMISSION SAVED SUCCESSFULLY
File: submission_v5_hubstats.csv


In [92]:
import os

for f in os.listdir("/kaggle/working"):
    print(f)

submission_v5_hubstats.csv
submission_v2.csv
submission_direct_catboost.csv
catboost_info
submission_v1_recursive.csv
.virtual_documents
submission_statistical_v2.csv


In [93]:
import pandas as pd
import numpy as np

v5 = pd.read_csv("/kaggle/working/submission_v5_hubstats.csv")
direct = pd.read_csv("/kaggle/working/submission_direct_catboost.csv")
v2 = pd.read_csv("/kaggle/working/submission_v2.csv")

print("V5:")
print(v5["OrderVolume"].describe())

print("\nDIRECT:")
print(direct["OrderVolume"].describe())

print("\nV2:")
print(v2["OrderVolume"].describe())

# Verify IDs match
print("\nID alignment:")
print("V5 vs Direct:", (v5["Id"].values == direct["Id"].values).mean())
print("V5 vs V2:", (v5["Id"].values == v2["Id"].values).mean())

V5:
count    46830.000000
mean      5980.814539
std       3619.327320
min          0.000000
25%       4180.268728
50%       5910.465767
75%       7937.820181
max      28783.330617
Name: OrderVolume, dtype: float64

DIRECT:
count    46830.000000
mean      6011.533374
std       3562.305586
min          0.000000
25%       4177.245435
50%       5995.842193
75%       8071.058928
max      29281.757790
Name: OrderVolume, dtype: float64

V2:
count    46830.000000
mean      2704.112247
std       1894.448852
min          0.000000
25%       1317.689301
50%       2641.672711
75%       3869.214943
max      12586.168323
Name: OrderVolume, dtype: float64

ID alignment:
V5 vs Direct: 1.0
V5 vs V2: 1.0


In [94]:
pred_df = pd.DataFrame({
    "v5": v5["OrderVolume"],
    "direct": direct["OrderVolume"],
    "v2": v2["OrderVolume"]
})

print(pred_df.corr())

print("\nMean absolute differences:")
print(pred_df.apply(lambda x: np.abs(x - pred_df["v5"]).mean()))

              v5    direct        v2
v5      1.000000  0.992321  0.732400
direct  0.992321  1.000000  0.724498
v2      0.732400  0.724498  1.000000

Mean absolute differences:
v5           0.000000
direct     285.426781
v2        3293.498375
dtype: float64


In [95]:
# ============================================================
# CREATE ENSEMBLE SUBMISSIONS
# ============================================================

def save_blend(name, prediction):
    
    prediction = np.clip(prediction, 0, None)
    
    sub = pd.DataFrame({
        "Id": v5["Id"],
        "OrderVolume": prediction
    })
    
    path = f"/kaggle/working/{name}.csv"
    sub.to_csv(path, index=False)
    
    print(f"{name} saved")
    print(sub["OrderVolume"].describe()[["mean", "std", "min", "max"]])
    print()


# ------------------------------------------------------------
# ARITHMETIC BLENDS: V5 + DIRECT
# ------------------------------------------------------------

save_blend(
    "submission_blend_90v5_10direct",
    0.90 * v5["OrderVolume"] +
    0.10 * direct["OrderVolume"]
)

save_blend(
    "submission_blend_80v5_20direct",
    0.80 * v5["OrderVolume"] +
    0.20 * direct["OrderVolume"]
)

save_blend(
    "submission_blend_70v5_30direct",
    0.70 * v5["OrderVolume"] +
    0.30 * direct["OrderVolume"]
)


# ------------------------------------------------------------
# LOG-SPACE BLEND
# ------------------------------------------------------------

log_blend = np.expm1(
    0.90 * np.log1p(v5["OrderVolume"]) +
    0.10 * np.log1p(direct["OrderVolume"])
)

save_blend(
    "submission_logblend_90v5_10direct",
    log_blend
)


# ------------------------------------------------------------
# SMALL DIVERSITY EXPERIMENT WITH V2
# ONLY 5% V2
# ------------------------------------------------------------

save_blend(
    "submission_blend_95v5_5v2",
    0.95 * v5["OrderVolume"] +
    0.05 * v2["OrderVolume"]
)


print("=" * 60)
print("ALL BLENDS CREATED")
print("=" * 60)

submission_blend_90v5_10direct saved
mean     5983.886422
std      3611.158601
min         0.000000
max     28833.173334
Name: OrderVolume, dtype: float64

submission_blend_80v5_20direct saved
mean     5986.958306
std      3603.529897
min         0.000000
max     28883.016051
Name: OrderVolume, dtype: float64

submission_blend_70v5_30direct saved
mean     5990.030190
std      3596.444646
min         0.000000
max     28932.858769
Name: OrderVolume, dtype: float64

submission_logblend_90v5_10direct saved
mean     5982.832536
std      3610.346607
min         0.000000
max     28832.789158
Name: OrderVolume, dtype: float64

submission_blend_95v5_5v2 saved
mean     5816.979424
std      3508.328508
min         0.000000
max     27938.693380
Name: OrderVolume, dtype: float64

ALL BLENDS CREATED


In [96]:
# ============================================================
# V8 SEASONAL SAME-DATE MODEL
# ============================================================

from catboost import CatBoostRegressor
import pandas as pd
import numpy as np

# ------------------------------------------------------------
# 1. COPY DATA
# ------------------------------------------------------------

v8_train = train.copy()
v8_test = test.copy()

v8_train["Date"] = pd.to_datetime(v8_train["Date"])
v8_test["Date"] = pd.to_datetime(v8_test["Date"])


# ------------------------------------------------------------
# 2. CALENDAR FEATURES
# ------------------------------------------------------------

def add_v8_calendar(df):

    df = df.copy()

    df["Year"] = df["Date"].dt.year
    df["Month"] = df["Date"].dt.month
    df["Day"] = df["Date"].dt.day
    df["DayOfYear"] = df["Date"].dt.dayofyear
    df["WeekOfYear"] = df["Date"].dt.isocalendar().week.astype(int)

    df["IsWeekend"] = (df["Weekday"] >= 6).astype(int)

    df["Month_sin"] = np.sin(
        2 * np.pi * df["Month"] / 12
    )

    df["Month_cos"] = np.cos(
        2 * np.pi * df["Month"] / 12
    )

    df["Weekday_sin"] = np.sin(
        2 * np.pi * df["Weekday"] / 7
    )

    df["Weekday_cos"] = np.cos(
        2 * np.pi * df["Weekday"] / 7
    )

    return df


v8_train = add_v8_calendar(v8_train)
v8_test = add_v8_calendar(v8_test)


# ------------------------------------------------------------
# 3. PREVIOUS YEAR SAME-DATE FEATURES
# ------------------------------------------------------------

season_lookup = train[
    ["HubID", "Date", "OrderVolume"]
].copy()

season_lookup["Date"] = pd.to_datetime(season_lookup["Date"])

# Create shifted date so 2014 demand maps to 2015 date
season_lookup["Date"] = (
    season_lookup["Date"] +
    pd.DateOffset(years=1)
)

season_lookup = season_lookup.rename(
    columns={
        "OrderVolume": "PrevYearSameDate"
    }
)

season_lookup = season_lookup[
    ["HubID", "Date", "PrevYearSameDate"]
]


# Merge into train and test
v8_train = v8_train.merge(
    season_lookup,
    on=["HubID", "Date"],
    how="left"
)

v8_test = v8_test.merge(
    season_lookup,
    on=["HubID", "Date"],
    how="left"
)


# ------------------------------------------------------------
# 4. PREVIOUS YEAR ±7 DAY WINDOW FEATURES
# ------------------------------------------------------------

# Use 2014 data only for seasonal statistics
hist_2014 = train.copy()
hist_2014["Date"] = pd.to_datetime(hist_2014["Date"])

hist_2014 = hist_2014[
    hist_2014["Date"].dt.year == 2014
].copy()

hist_2014["Month"] = hist_2014["Date"].dt.month
hist_2014["Day"] = hist_2014["Date"].dt.day

season_md = (
    hist_2014
    .groupby(["HubID", "Month", "Day"])["OrderVolume"]
    .mean()
    .reset_index()
    .rename(columns={"OrderVolume": "PrevYearExact"})
)

v8_train = v8_train.merge(
    season_md,
    on=["HubID", "Month", "Day"],
    how="left"
)

v8_test = v8_test.merge(
    season_md,
    on=["HubID", "Month", "Day"],
    how="left"
)


# ------------------------------------------------------------
# 5. HUB RECENT DEMAND STATISTICS
# ------------------------------------------------------------

hub_stats = (
    train
    .groupby("HubID")["OrderVolume"]
    .agg([
        "mean",
        "median",
        "std",
        "min",
        "max"
    ])
    .reset_index()
)

hub_stats.columns = [
    "HubID",
    "HubMean",
    "HubMedian",
    "HubStd",
    "HubMin",
    "HubMax"
]

v8_train = v8_train.merge(
    hub_stats,
    on="HubID",
    how="left"
)

v8_test = v8_test.merge(
    hub_stats,
    on="HubID",
    how="left"
)


# ------------------------------------------------------------
# 6. METADATA
# ------------------------------------------------------------

meta = pd.read_csv(
    "/kaggle/input/competitions/ch-27-celebal-technologies-nit-rourkela/hub_metadata.csv"
)

for df_name in ["v8_train", "v8_test"]:

    df = globals()[df_name]

    missing_meta = [
        c for c in meta.columns
        if c != "HubID" and c not in df.columns
    ]

    if missing_meta:

        df = df.merge(
            meta[["HubID"] + missing_meta],
            on="HubID",
            how="left"
        )

    globals()[df_name] = df


# ------------------------------------------------------------
# 7. FEATURE ENGINEERING
# ------------------------------------------------------------

# Important ratios
for df_name in ["v8_train", "v8_test"]:

    df = globals()[df_name]

    df["SeasonToHubRatio"] = (
        df["PrevYearSameDate"] /
        (df["HubMean"] + 1)
    )

    df["SeasonToMedianRatio"] = (
        df["PrevYearSameDate"] /
        (df["HubMedian"] + 1)
    )

    globals()[df_name] = df


# ------------------------------------------------------------
# 8. FEATURES
# ------------------------------------------------------------

DROP_V8 = [
    "Date",
    "OrderVolume",
    "Id",
    "AppSessions"
]

V8_FEATURES = [
    c for c in v8_train.columns
    if c not in DROP_V8
    and c in v8_test.columns
]

V8_CAT = [
    c for c in [
        "HubID",
        "HubFormat",
        "AssortmentTier",
        "LoyaltyProgram",
        "LoyaltyProgramInterval"
    ]
    if c in V8_FEATURES
]


# ------------------------------------------------------------
# 9. CLEAN DATA
# ------------------------------------------------------------

for col in V8_FEATURES:

    if col in V8_CAT:

        v8_train[col] = (
            v8_train[col]
            .fillna("Missing")
            .astype(str)
        )

        v8_test[col] = (
            v8_test[col]
            .fillna("Missing")
            .astype(str)
        )

    else:

        v8_train[col] = (
            v8_train[col]
            .replace([np.inf, -np.inf], np.nan)
            .fillna(-999)
        )

        v8_test[col] = (
            v8_test[col]
            .replace([np.inf, -np.inf], np.nan)
            .fillna(-999)
        )


print("=" * 60)
print("V8 DATA READY")
print("=" * 60)

print("Features:", len(V8_FEATURES))
print("Categorical:", V8_CAT)

print(
    "PrevYearSameDate available in test:",
    (v8_test["PrevYearSameDate"] != -999).mean()
)


# ------------------------------------------------------------
# 10. TRAIN FAST MODEL
# ------------------------------------------------------------

v8_model = CatBoostRegressor(
    iterations=550,
    depth=8,
    learning_rate=0.08,
    l2_leaf_reg=6,
    loss_function="RMSE",
    random_seed=2026,
    verbose=100,
    thread_count=-1
)

v8_model.fit(
    v8_train[V8_FEATURES],
    np.log1p(v8_train["OrderVolume"]),
    cat_features=V8_CAT
)


# ------------------------------------------------------------
# 11. PREDICT
# ------------------------------------------------------------

v8_pred = np.expm1(
    v8_model.predict(
        v8_test[V8_FEATURES]
    )
)

v8_pred = np.clip(v8_pred, 0, None)

# Closed = zero
v8_pred[
    v8_test["IsOpen"].values == 0
] = 0


# ------------------------------------------------------------
# 12. SAVE PURE V8
# ------------------------------------------------------------

submission_v8 = pd.DataFrame({
    "Id": test["Id"],
    "OrderVolume": v8_pred
})

submission_v8.to_csv(
    "/kaggle/working/submission_v8_seasonal.csv",
    index=False
)


# ------------------------------------------------------------
# 13. CREATE V5 + V8 ENSEMBLE
# ------------------------------------------------------------

v5_best = pd.read_csv(
    "/kaggle/working/submission_v5_hubstats.csv"
)

# Log-space blend - V5 dominant
blend_v5_v8 = np.expm1(
    0.70 * np.log1p(v5_best["OrderVolume"].values)
    +
    0.30 * np.log1p(v8_pred)
)

blend_v5_v8 = np.clip(blend_v5_v8, 0, None)

blend_v5_v8[
    test["IsOpen"].values == 0
] = 0


submission_v8_blend = pd.DataFrame({
    "Id": test["Id"],
    "OrderVolume": blend_v5_v8
})

submission_v8_blend.to_csv(
    "/kaggle/working/submission_v8blend_70v5_30seasonal.csv",
    index=False
)


print("\n" + "=" * 60)
print("V8 COMPLETE")
print("=" * 60)

print("\nV8 prediction stats:")
print(pd.Series(v8_pred).describe())

print("\nV5 + V8 blend stats:")
print(pd.Series(blend_v5_v8).describe())

print("\nFiles created:")
print("submission_v8_seasonal.csv")
print("submission_v8blend_70v5_30seasonal.csv")

V8 DATA READY
Features: 34
Categorical: ['HubID', 'HubFormat', 'AssortmentTier', 'LoyaltyProgram', 'LoyaltyProgramInterval']
PrevYearSameDate available in test: 0.8808456117873158
0:	learn: 3.0589692	total: 751ms	remaining: 6m 52s
100:	learn: 0.1365430	total: 1m 2s	remaining: 4m 39s
200:	learn: 0.1195264	total: 2m 6s	remaining: 3m 40s
300:	learn: 0.1108112	total: 3m 13s	remaining: 2m 39s
400:	learn: 0.1062839	total: 4m 17s	remaining: 1m 35s
500:	learn: 0.1024866	total: 5m 22s	remaining: 31.5s
549:	learn: 0.1011391	total: 5m 53s	remaining: 0us

V8 COMPLETE

V8 prediction stats:
count    46830.000000
mean      5964.779546
std       3539.116065
min          0.000000
25%       4214.426372
50%       5955.474726
75%       7934.000321
max      26605.729680
dtype: float64

V5 + V8 blend stats:
count    46830.000000
mean      5972.120314
std       3584.061695
min          0.000000
25%       4196.164041
50%       5924.619549
75%       7929.667614
max      27730.210867
dtype: float64

Files creat

In [97]:
import pandas as pd
import numpy as np
import os

PATH = "/kaggle/working"

# Load predictions
v5 = pd.read_csv(f"{PATH}/submission_v5_hubstats.csv")
direct = pd.read_csv(f"{PATH}/submission_direct_catboost.csv")

print("Files loaded")
print(v5.shape, direct.shape)

# Verify IDs
assert (v5["Id"].values == direct["Id"].values).all()

# Different log-space blends
weights = [0.55, 0.60, 0.65, 0.70, 0.75, 0.80, 0.85, 0.90, 0.95]

for w in weights:
    pred = np.expm1(
        w * np.log1p(v5["OrderVolume"].values) +
        (1-w) * np.log1p(direct["OrderVolume"].values)
    )

    pred = np.clip(pred, 0, None)

    sub = pd.DataFrame({
        "Id": v5["Id"],
        "OrderVolume": pred
    })

    filename = f"submission_blend_v5_{int(w*100)}_direct_{int((1-w)*100)}.csv"
    sub.to_csv(f"{PATH}/{filename}", index=False)

print("\nALL BLENDS CREATED:")
for f in sorted(os.listdir(PATH)):
    if "submission_blend" in f:
        print(f)

Files loaded
(46830, 2) (46830, 2)

ALL BLENDS CREATED:
submission_blend_70v5_30direct.csv
submission_blend_80v5_20direct.csv
submission_blend_90v5_10direct.csv
submission_blend_95v5_5v2.csv
submission_blend_v5_55_direct_44.csv
submission_blend_v5_60_direct_40.csv
submission_blend_v5_65_direct_35.csv
submission_blend_v5_70_direct_30.csv
submission_blend_v5_75_direct_25.csv
submission_blend_v5_80_direct_19.csv
submission_blend_v5_85_direct_15.csv
submission_blend_v5_90_direct_9.csv
submission_blend_v5_95_direct_5.csv


In [98]:
import pandas as pd
import numpy as np

PATH = "/kaggle/working"

v5 = pd.read_csv(f"{PATH}/submission_v5_hubstats.csv")
direct = pd.read_csv(f"{PATH}/submission_direct_catboost.csv")
v8 = pd.read_csv(f"{PATH}/submission_v8_seasonal.csv")

assert (v5.Id.values == direct.Id.values).all()
assert (v5.Id.values == v8.Id.values).all()

# LOG SPACE ENSEMBLE
# V5 remains dominant because it is our strongest model
pred = np.expm1(
    0.75 * np.log1p(v5["OrderVolume"].values) +
    0.15 * np.log1p(direct["OrderVolume"].values) +
    0.10 * np.log1p(v8["OrderVolume"].values)
)

pred = np.clip(pred, 0, None)

submission = pd.DataFrame({
    "Id": v5["Id"],
    "OrderVolume": pred
})

submission.to_csv(
    f"{PATH}/submission_final_3model_blend.csv",
    index=False
)

print(submission.shape)
print(submission["OrderVolume"].describe())
print("Saved: submission_final_3model_blend.csv")

(46830, 2)
count    46830.000000
mean      5980.870275
std       3593.900332
min          0.000000
25%       4186.669282
50%       5929.075108
75%       7956.166661
max      28501.223641
Name: OrderVolume, dtype: float64
Saved: submission_final_3model_blend.csv


In [99]:
import pandas as pd
import numpy as np

PATH = "/kaggle/working"

v5 = pd.read_csv(f"{PATH}/submission_v5_hubstats.csv")

# Very small calibration candidates
# Based on RMSLE, tiny systematic over/under prediction adjustments can matter

for factor in [0.97, 0.98, 1.02, 1.03]:
    
    pred = v5["OrderVolume"].values * factor
    pred = np.clip(pred, 0, None)
    
    sub = pd.DataFrame({
        "Id": v5["Id"],
        "OrderVolume": pred
    })
    
    fname = f"submission_v5_scale_{factor:.2f}.csv"
    sub.to_csv(f"{PATH}/{fname}", index=False)

print("Created calibration submissions")

Created calibration submissions


In [100]:
import pandas as pd
import numpy as np

PATH = "/kaggle/working"

v5 = pd.read_csv(f"{PATH}/submission_v5_hubstats.csv")

for factor in [0.975, 0.985]:
    pred = np.clip(
        v5["OrderVolume"].values * factor,
        0,
        None
    )

    pd.DataFrame({
        "Id": v5["Id"],
        "OrderVolume": pred
    }).to_csv(
        f"{PATH}/submission_v5_scale_{factor}.csv",
        index=False
    )

print("Created:")
print("submission_v5_scale_0.975.csv")
print("submission_v5_scale_0.985.csv")

Created:
submission_v5_scale_0.975.csv
submission_v5_scale_0.985.csv


In [101]:
import pandas as pd
import numpy as np

PATH = "/kaggle/working"

v5 = pd.read_csv(f"{PATH}/submission_v5_hubstats.csv")

factor = 0.982

submission = pd.DataFrame({
    "Id": v5["Id"],
    "OrderVolume": np.clip(
        v5["OrderVolume"] * factor,
        0,
        None
    )
})

submission.to_csv(
    f"{PATH}/submission_v5_scale_0.982.csv",
    index=False
)

print("Ready!")

Ready!
